# RA-MDST — Revision Experiments (Frontiers in AI, revision 2 — notebook v4)

This one notebook regenerates **every table and figure** needed for the revised manuscript and the response letter.
It starts from the code in `main (1).ipynb` and `main (2).ipynb` (the MDST and RA-MDST samplers are copied with the same random-number order),
so **repeat 0 of Protocol B and seed 42 of Protocol A reproduce the numbers from the original submission**.

| Section | Reviewer comment it answers |
|---|---|
| 1. Neighborhood-overlap audit | R1: overlap 0.7582 vs 0.7554; R2 #5 |
| 2. Protocol A reproduction + leakage diagnostics | R1/R2: reframe around Protocol B; 93.11 / 92.41 / 92.76 inconsistency; Table 2 |
| 3. Protocol B benchmark (10 repeats × 5 folds, 10 classifiers) | R1: SMOTE, Borderline-SMOTE, ADASYN, ROS baselines; protocol clarity; repetitions |
| 4. Statistics | R1: 5-fold limitation, multiple-comparison correction, effect sizes, CIs |
| 5. Ablation (Protocol B) | R1/R2: what each RA-MDST component does under the leakage-controlled protocol |
| 6. Type validity | R1: type-preserving generation; proportion of invalid synthetic values |
| 7. Diversity metrics | R1: inter-sample distance, coverage, duplicates, class separability |
| 8. Sensitivity analysis | R1 + R2 #3: k, λ, number of RFE features, C |
| 9. Nested cross-validation | R1 + R2 #3 |
| 10. Partial balancing | R1: partial balancing and real-to-synthetic ratios |
| 11. Ordinal analysis | R1: ordinal nature of the target |
| 12. SHAP | R1: substantiate "explainability" or remove it |
| 13. External benchmarks | R2 #2: additional datasets |
| 16. Type-preserving SMOTE / Borderline-SMOTE / ADASYN | Round 3: separates type preservation from the redundancy correction |
| 17. Near-duplicate threshold | Round 4: sensitivity of the near-duplicate rate to its threshold (Table S5) |
| 15. Article figures | All result figures of the article, drawn from the tables (round 2: regenerate identically) |
| 14. Summary + MANIFEST + ZIP | Reproducibility: summary, table hashes for run-to-run comparison, results zip |

**v4 (round-2 revision):** Section 4 now also tests every method against no oversampling and writes
`T04c_holm_survival_summary.csv` (which comparisons survive Holm correction); Section 15 draws the article figures;
Section 14 writes `MANIFEST.csv`. Outside Kaggle, run everything with `python run_all.py --data <csv>`.
To update a finished v3 run quickly: add the previous version's Output as an Input — Sections 1–3 and 5–13 are
reused from their checkpoints and only Section 4, 15 and 14 are recomputed (about 2 minutes).

## How to run on Kaggle
1. **File → Import notebook** and upload this `.ipynb`.
2. **Add Input** → add your dataset that contains `NN_MissingHandlingdataset.csv` (the notebook finds it automatically anywhere under `/kaggle/input`).
3. Settings: **Internet ON** (only needed for Section 13 and to install `mord`; everything else works offline). CPU is enough — no GPU needed.
4. **Use "Save Version → Save & Run All (Commit)"**, not the interactive editor: it runs in the background (up to 12 h),
   survives closing the browser, and keeps the output files. Interactive sessions stop when idle and lose `/kaggle/working`.
5. Run `RUN_MODE = "QUICK"` first (≈5–10 min) to check everything works, then set `RUN_MODE = "FULL"` and **Run All** (≈1–2 h on Kaggle CPU).
6. Download `/kaggle/working/RA_MDST_revision_results.zip` from the Output panel. `RESULTS_SUMMARY.md` inside it lists the key numbers.

Every section is wrapped so a failure in one section is logged to `ERRORS.txt` and the rest still run.
Finished sections are checkpointed: if a run stops, re-run and it continues from the first unfinished section
(to continue in a new Kaggle session, add the previous version's Output as an Input).

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION (the only cell you normally edit)
# ============================================================
RUN_MODE = "FULL"          # "QUICK" = fast smoke test, "FULL" = numbers for the paper

SEED = 42
TARGET = "target"
DATA_FILE_HINT = "NN_MissingHandlingdataset"   # file name the loader looks for
DATA_PATH_OVERRIDE = None                       # or e.g. "/kaggle/input/avcrashold/NN_MissingHandlingdataset.csv"

# Predictors to exclude before any analysis (leakage check / identifier-like codes).
# Suggested robustness run:
#   DROP_COLUMNS = ["P1INJURED", "AccDETAILS", "AccidentDATE", "AccidentTIME", "AccidentADDRESS"]
# plus any other column that records injuries/casualties (the outcome itself) — see DROP_PATTERN.
DROP_COLUMNS = []
DROP_PATTERN = None                              # regex, e.g. r"INJUR|KILL|FATAL" to drop every outcome-like column

# Class codes exactly as in the original notebooks (CLASS_NAMES list in main (1).ipynb).
# >>> VERIFY against the raw DMV labels before submission. <<<
CLASS_NAMES = {0: "Minor", 1: "Moderate", 2: "None"}
# Severity order from least to most severe (used for the ordinal analysis, MAE and QWK).
SEVERITY_ORDER = ["None", "Minor", "Moderate"]

# Paper defaults (unchanged from the submission)
N_RFE = 100
RFE_STEP = 5
K_NEIGH = 3
LAM = (0.30, 0.70)
C_SVM = 1.0
IMB_K = 5                  # k for SMOTE / Borderline-SMOTE / ADASYN / SMOTE-NC (library default)

# Type handling for type-preserving RA-MDST (RA-MDST-TP)
ORDINAL_NAME_PATTERN = r"year"      # integer columns whose names match are treated as ordered numbers (interpolate + snap)
ORDINAL_COLUMNS_EXTRA = []          # add column names here if other integer columns are genuinely ordinal
MIN_PREFIX = 6                      # minimum shared name prefix for detecting mutually-exclusive binary groups

N_JOBS = -1

# Safety caps for the SVM solver (libsvm). Without a cap a non-converging fit (e.g. C = 10) can run for hours.
SVC_MAX_ITER = 1_000_000        # main benchmark (converged fits are unaffected)
FAST_SVC_MAX_ITER = 50_000      # inside the sensitivity and nested-CV loops
TASK_TIMEOUT_SEC = 3600         # any single parallel fold task taking longer than this aborts its section (logged), notebook continues

# Resume support: every finished section is saved to OUT_DIR/checkpoints and re-used on the next run.
# On Kaggle you can also add a previous version's Output as an Input — checkpoints found there are re-used too.
RESUME = True
SECTIONS_TO_RUN = list(range(1, 15)) + [16, 17]   # e.g. [8, 9, 10, 11, 12, 13, 14] to run only the remaining sections

import os
# Command-line / run_all.py overrides (environment variables); on Kaggle these are normally unset.
RUN_MODE = os.environ.get("RA_MDST_MODE", RUN_MODE)
DATA_PATH_OVERRIDE = os.environ.get("RA_MDST_DATA") or DATA_PATH_OVERRIDE
if os.environ.get("RA_MDST_SECTIONS"):
    SECTIONS_TO_RUN = [int(x) for x in os.environ["RA_MDST_SECTIONS"].split(",") if x.strip()]
if os.environ.get("RA_MDST_RESUME"):
    RESUME = os.environ["RA_MDST_RESUME"] == "1"
OUT_DIR = os.environ.get("RA_MDST_OUT") or ("/kaggle/working/revision_results" if os.path.isdir("/kaggle/working")
                                           else os.path.abspath("revision_results"))

if RUN_MODE == "QUICK":
    MAIN_REPEATS = 2
    TREE_EST, GB_EST, MLP_ITER = 50, 30, 300
    PROTO_A_SEEDS = [42, 0, 1]
    SENS_REPEATS = 1
    SENS_K = [3, 5]
    SENS_LAM = [(0.0, 1.0), (0.3, 0.7)]
    SENS_N = [50, 100]
    SENS_C = [0.1, 1.0]
    NESTED_OUTER_REPEATS, NESTED_INNER = 1, 2
    NESTED_GRID = dict(n=[50, 100], C=[0.1, 1.0], k=[3, 5], lam=[(0.3, 0.7)])
    PB_REPEATS, PB_ALPHAS = 1, [0.5, 1.0]
    ORD_REPEATS = 1
    SHAP_ENABLED = True
    EXT_REPEATS = 1
    EXT_DATASETS = ["ecoli"]
else:
    MAIN_REPEATS = 10
    TREE_EST, GB_EST, MLP_ITER = 300, 150, 1000
    PROTO_A_SEEDS = [42] + list(range(10))
    SENS_REPEATS = 3
    SENS_K = [2, 3, 5, 7, 10]
    SENS_LAM = [(0.0, 1.0), (0.1, 0.9), (0.2, 0.8), (0.3, 0.7), (0.4, 0.6), (0.5, 0.5)]
    SENS_N = [25, 50, 75, 100, 150, 189]
    SENS_C = [0.01, 0.1, 1.0, 10.0, 100.0]
    NESTED_OUTER_REPEATS, NESTED_INNER = 2, 3
    NESTED_GRID = dict(n=[25, 50, 100, 150, 189], C=[0.01, 0.1, 1.0, 10.0], k=[3, 5, 7],
                       lam=[(0.0, 1.0), (0.2, 0.8), (0.3, 0.7)])
    PB_REPEATS, PB_ALPHAS = 5, [0.25, 0.5, 0.75, 1.0]
    ORD_REPEATS = 5
    SHAP_ENABLED = True
    EXT_REPEATS = 3
    EXT_DATASETS = ["ecoli", "yeast_me2", "car_eval_34", "us_crime", "abalone", "wine_quality"]

N_SPLITS = 5
SAMPLERS_MAIN = ["None", "ROS", "SMOTE", "Borderline-SMOTE", "ADASYN", "SMOTE-NC",
                 "MDST", "RA-MDST", "RA-MDST-TP"]
ABLATION_SAMPLERS = ["Abl-ExclusiveOnly", "Abl-InterpOnly"]   # Original MDST = "MDST", Full RA-MDST = "RA-MDST"
CLASSIFIERS = ["LogisticRegression", "LinearSVM", "RBFSVM", "KNN", "DecisionTree",
               "RandomForest", "ExtraTrees", "GradientBoosting", "AdaBoost", "MLP"]
ABLATION_CLASSIFIERS = ["LinearSVM", "LogisticRegression", "RandomForest"]
PB_SAMPLERS = ["ROS", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]
PB_CLASSIFIERS = ["LinearSVM", "LogisticRegression", "RandomForest"]
ORD_SAMPLERS = ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]
NESTED_SAMPLERS = ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]

print("RUN_MODE:", RUN_MODE, "| main design:", MAIN_REPEATS, "x", N_SPLITS, "folds | output:", OUT_DIR)

In [ ]:
# ============================================================
# CELL 2 — Imports, installs, versions
# ============================================================
import sys, re, glob, json, time, math, zipfile, warnings, platform, traceback, subprocess, importlib
from collections import defaultdict
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

def _ensure(pkg, pip_name=None):
    try:
        return importlib.import_module(pkg)
    except Exception:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg],
                           check=True, capture_output=True, timeout=300)
            return importlib.import_module(pkg)
        except Exception as e:
            print(f"[warn] could not import/install {pkg}: {e}")
            return None

imblearn = _ensure("imblearn", "imbalanced-learn")
shap = _ensure("shap") if SHAP_ENABLED else None
mord = None
try:
    mord = importlib.import_module("mord")
except Exception:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mord"], check=True,
                       capture_output=True, timeout=180)
        mord = importlib.import_module("mord")
    except Exception:
        mord = None

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats
from scipy.spatial.distance import cdist
import sklearn
from joblib import Parallel, delayed

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
                             confusion_matrix, pairwise_distances, silhouette_score)
from imblearn.over_sampling import (RandomOverSampler, SMOTE, BorderlineSMOTE, ADASYN,
                                    SMOTENC, SMOTEN)

os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures"); TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)

VERSIONS = {
    "python": platform.python_version(), "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__, "imbalanced-learn": imblearn.__version__,
    "matplotlib": matplotlib.__version__,
    "shap": getattr(shap, "__version__", "not available"),
    "mord": getattr(mord, "__version__", "installed" if mord else "not available"),
    "run_mode": RUN_MODE, "seed": SEED,
}
json.dump(VERSIONS, open(os.path.join(OUT_DIR, "environment_versions.json"), "w"), indent=2)
with open(os.path.join(OUT_DIR, "requirements.txt"), "w") as fh:
    for k in ["numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "matplotlib", "shap"]:
        if VERSIONS[k] not in ("not available",):
            fh.write(f"{k}=={VERSIONS[k]}\n")
print(json.dumps(VERSIONS, indent=1))

ERRORS = []
SUMMARY = {}           # key numbers collected for RESULTS_SUMMARY.md

CKPT_DIR = os.path.join(OUT_DIR, "checkpoints"); os.makedirs(CKPT_DIR, exist_ok=True)

SECTION_CODE_VERSION = {4: 3}   # v2 of Section 4 adds the no-oversampling comparisons, Holm summary and 5-panel forest plot

def _find_checkpoint(fname):
    p = os.path.join(CKPT_DIR, fname)
    if os.path.exists(p):
        return p
    hits = glob.glob(f"/kaggle/input/**/checkpoints/{fname}", recursive=True)
    return hits[0] if hits else None

def run_section(name, fn):
    """Run one section; log any exception and continue with the rest of the notebook.
    Finished sections are checkpointed so a re-run can skip them."""
    t0 = time.time()
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    num = int(re.search(r"Section (\d+)", name).group(1))
    cfg = json.dumps({k: str(globals().get(k)) for k in [
        "RUN_MODE", "SEED", "N_RFE", "RFE_STEP", "K_NEIGH", "LAM", "C_SVM", "IMB_K", "CLASS_NAMES", "SEVERITY_ORDER",
        "ORDINAL_NAME_PATTERN", "ORDINAL_COLUMNS_EXTRA", "MAIN_REPEATS", "PROTO_A_SEEDS", "SENS_REPEATS", "SENS_K", "SENS_LAM",
        "SENS_N", "SENS_C", "NESTED_OUTER_REPEATS", "NESTED_INNER", "NESTED_GRID", "PB_REPEATS", "PB_ALPHAS", "ORD_REPEATS",
        "EXT_REPEATS", "EXT_DATASETS", "EXTRA_CSV", "N_OBS", "N_FEAT", "TREE_EST", "GB_EST", "DROP_COLUMNS", "DROP_PATTERN"]}, sort_keys=True)
    import hashlib
    ver = SECTION_CODE_VERSION.get(num, 1)          # bumped when a section's code changes, so old checkpoints are not reused
    if ver > 1:
        cfg += f"|v{ver}"
    fname = f"section{num:02d}_{RUN_MODE}_{hashlib.md5(cfg.encode()).hexdigest()[:8]}.pkl"
    if num != 14:
        ck = _find_checkpoint(fname) if (RESUME or num not in SECTIONS_TO_RUN) else None
        if ck:
            out, summ = pd.read_pickle(ck)
            SUMMARY.update(summ)
            if os.path.dirname(ck) != CKPT_DIR:
                pd.to_pickle((out, summ), os.path.join(CKPT_DIR, fname))
                # bring that run's tables and figures into this output folder (needed by Sections 15 and 14)
                import shutil
                prev = os.path.dirname(os.path.dirname(ck))
                for sub, dst in [("tables", TAB_DIR), ("figures", FIG_DIR)]:
                    for f_ in glob.glob(os.path.join(prev, sub, "*")):
                        if not os.path.exists(os.path.join(dst, os.path.basename(f_))):
                            shutil.copy2(f_, dst)
            print(f"[resumed] {name} loaded from checkpoint {ck} (tables/figures from that run are in its output folder)")
            return out
        if num not in SECTIONS_TO_RUN:
            print(f"[skipped] {name} (not in SECTIONS_TO_RUN, no checkpoint)")
            return None
    try:
        before = set(SUMMARY)
        out = fn()
        if num != 14 and out is not None:
            pd.to_pickle((out, {k: SUMMARY[k] for k in SUMMARY if k not in before}), os.path.join(CKPT_DIR, fname))
        print(f"[done] {name} in {time.time()-t0:.1f}s")
        return out
    except Exception:
        tb = traceback.format_exc()
        ERRORS.append(f"## {name}\n{tb}")
        print(f"[ERROR] {name} failed — logged, continuing.\n{tb}")
        with open(os.path.join(OUT_DIR, "ERRORS.txt"), "w") as fh:
            fh.write("\n\n".join(ERRORS))
        return None

def save_table(df, name, index=False):
    path = os.path.join(TAB_DIR, name)
    df.to_csv(path, index=index)
    return path

# ---------- plotting style (colour-blind-safe, fixed colour per sampler) ----------
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "legend.frameon": False, "font.family": "DejaVu Sans"})
SAMPLER_COLORS = {
    "None": "#8c8c8c", "ROS": "#e87ba4", "SMOTE": "#eb6834", "Borderline-SMOTE": "#eda100",
    "ADASYN": "#e34948", "SMOTE-NC": "#008300", "MDST": "#1baf7a",
    "RA-MDST": "#2a78d6", "RA-MDST-TP": "#6250d6",
    "Abl-ExclusiveOnly": "#86b6ef", "Abl-InterpOnly": "#184f95",
}
def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")
    plt.show(); plt.close(fig)

In [ ]:
# ============================================================
# CELL 3 — Load data and infer variable types
# ============================================================
def find_data():
    if DATA_PATH_OVERRIDE:
        return DATA_PATH_OVERRIDE
    pats = ["/kaggle/input/**/*.csv", "/kaggle/working/*.csv", "/content/*.csv", "/mnt/data/*.csv", "./*.csv", "./data/*.csv"]
    files = []
    for p in pats:
        files += glob.glob(p, recursive=True)
    pref = [f for f in files if DATA_FILE_HINT in os.path.basename(f)]
    if pref:
        return pref[0]
    raise FileNotFoundError(f"Could not find a CSV containing '{DATA_FILE_HINT}'. "
                            "On Kaggle: Add Input → your dataset. Or set DATA_PATH_OVERRIDE.")

DATA_PATH = find_data()
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in df.columns if str(c).lower().startswith("unnamed:")], errors="ignore")
X_df = df.drop(columns=[TARGET]).copy()
_drop = [c for c in X_df.columns if c in DROP_COLUMNS or (DROP_PATTERN and re.search(DROP_PATTERN, str(c), re.I))]
missing = [c for c in DROP_COLUMNS if c not in X_df.columns]
if missing:
    print("[warn] DROP_COLUMNS not found in data:", missing)
X_df = X_df.drop(columns=_drop)
print("Dropped predictors:", _drop if _drop else "none")
for c in X_df.columns:
    X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
assert not X_df.isna().any().any(), "Missing / non-numeric values in predictors."
FEATURES = list(X_df.columns)
X_RAW = X_df.to_numpy(dtype=float)
Y = df[TARGET].to_numpy(dtype=int)
CLASSES = np.unique(Y)
ORD_MAP = {c: SEVERITY_ORDER.index(CLASS_NAMES[c]) for c in CLASSES}
N_OBS, N_FEAT = X_RAW.shape
print("Loaded:", DATA_PATH, "| shape:", X_RAW.shape)
print("Class counts:", {f"{c} ({CLASS_NAMES[c]})": int((Y == c).sum()) for c in CLASSES})
print("Ordinal coding used in Section 11:", {CLASS_NAMES[c]: ORD_MAP[c] for c in CLASSES})


def infer_kinds(Xr, names):
    kinds = []
    for j, nm in enumerate(names):
        v = Xr[:, j]
        is_int = np.all(np.abs(v - np.round(v)) < 1e-9)
        u = np.unique(v)
        if is_int and set(u.tolist()).issubset({0.0, 1.0}):
            kinds.append("binary")
        elif is_int and (re.search(ORDINAL_NAME_PATTERN, nm, re.I) or nm in ORDINAL_COLUMNS_EXTRA):
            kinds.append("ordinal")
        elif is_int:
            kinds.append("nominal")      # label-encoded categorical (manufacturer, city, ZIP, ...)
        else:
            kinds.append("continuous")
    return np.array(kinds)


def detect_exclusive_groups(Xr, names, kinds):
    """Binary columns that share a name prefix and are never co-active (row sum <= 1) are
    treated as one categorical block (e.g. one-hot dummies). Detected from training rows only."""
    bin_idx = [j for j, k in enumerate(kinds) if k == "binary"]
    pref = defaultdict(list)
    for j in bin_idx:
        nm = str(names[j])
        for L in range(MIN_PREFIX, len(nm)):
            pref[nm[:L]].append(j)
    cands = []
    for p, cols in pref.items():
        if len(cols) < 2:
            continue
        s = Xr[:, cols].sum(1)
        if s.max() <= 1:
            cands.append((len(cols), len(p), p, tuple(cols), bool(np.all(s == 1))))
    cands.sort(key=lambda t: (-t[0], -t[1]))
    used, groups = set(), []
    for _, _, p, cols, exact in cands:
        if any(c in used for c in cols):
            continue
        groups.append({"prefix": p, "cols": list(cols), "exact": exact})
        used.update(cols)
    return groups


KINDS_ALL = infer_kinds(X_RAW, FEATURES)
GROUPS_ALL = detect_exclusive_groups(X_RAW, FEATURES, KINDS_ALL)
kind_tab = pd.DataFrame({"feature": FEATURES, "kind": KINDS_ALL,
                         "n_unique": [len(np.unique(X_RAW[:, j])) for j in range(N_FEAT)]})
save_table(kind_tab, "T00_feature_types.csv")
save_table(pd.DataFrame([{"prefix": g["prefix"], "n_cols": len(g["cols"]), "exactly_one": g["exact"],
                          "columns": "; ".join(FEATURES[c] for c in g["cols"])} for g in GROUPS_ALL]),
           "T00_exclusive_binary_groups.csv")
print("Variable kinds:", kind_tab["kind"].value_counts().to_dict())
print("Mutually-exclusive binary groups detected:", len(GROUPS_ALL))
print(kind_tab[kind_tab.kind != "binary"].to_string(index=False))
SUMMARY["data"] = {"path": DATA_PATH, "n": int(N_OBS), "p": int(N_FEAT), "dropped_predictors": _drop,
                   "class_counts": {CLASS_NAMES[c]: int((Y == c).sum()) for c in CLASSES},
                   "kinds": kind_tab["kind"].value_counts().to_dict(), "n_groups": len(GROUPS_ALL)}

In [ ]:
# ============================================================
# CELL 4 — Fold-internal preprocessing (scaler + RFE) and type information
# ============================================================
class TypeInfo:
    """Valid values of each selected column, in the *scaled* space, learned from real training rows."""
    def __init__(self, Xr_train, scaler_mean, scaler_scale, kinds, groups_full, sel):
        self.sel = np.asarray(sel)
        self.kinds = kinds[self.sel]
        self.mean = scaler_mean[self.sel]; self.scale = scaler_scale[self.sel]
        self.p = len(self.sel)
        Xs = Xr_train[:, self.sel]
        self.valid_raw = [np.unique(Xs[:, j]) for j in range(self.p)]
        self.valid_s = [(v - self.mean[j]) / self.scale[j] for j, v in enumerate(self.valid_raw)]
        self.lo = Xs.min(0); self.hi = Xs.max(0)
        pos = {c: i for i, c in enumerate(self.sel)}
        self.groups = []
        for g in groups_full:
            cols = [pos[c] for c in g["cols"] if c in pos]
            if len(cols) >= 2:
                self.groups.append({"cols": np.array(cols), "exact": g["exact"] and len(cols) == len(g["cols"])})
        self.disc_idx = np.where(np.isin(self.kinds, ["binary", "nominal"]))[0]
        self.ord_idx = np.where(self.kinds == "ordinal")[0]
        self.cont_idx = np.where(self.kinds == "continuous")[0]
        self.cat_idx = self.disc_idx                      # for SMOTE-NC


def rfe_support(Xs_train, y_train, n, seed):
    if n >= Xs_train.shape[1]:
        return np.arange(Xs_train.shape[1])
    est = LinearSVC(C=1.0, dual=False, max_iter=15000, random_state=seed)
    sel = RFE(estimator=est, n_features_to_select=n, step=RFE_STEP).fit(Xs_train, y_train)
    return np.where(sel.support_)[0]


class FoldCtx:
    """Everything fitted on the REAL training rows of one outer fold only."""
    def __init__(self, Xr, y, tr, te, seed, names=None):
        names = FEATURES if names is None else names
        self.tr, self.te, self.seed = tr, te, seed
        self.Xr_tr, self.Xr_te = Xr[tr], Xr[te]
        self.y_tr, self.y_te = y[tr], y[te]
        self.scaler = StandardScaler().fit(self.Xr_tr)
        self.Xs_tr = self.scaler.transform(self.Xr_tr); self.Xs_te = self.scaler.transform(self.Xr_te)
        self.kinds = infer_kinds(self.Xr_tr, names)
        self.groups = detect_exclusive_groups(self.Xr_tr, names, self.kinds)
        self._sel, self._ti = {}, {}

    def sel(self, n):
        if n not in self._sel:
            self._sel[n] = rfe_support(self.Xs_tr, self.y_tr, n, self.seed)
        return self._sel[n]

    def data(self, n):
        s = self.sel(n)
        if n not in self._ti:
            self._ti[n] = TypeInfo(self.Xr_tr, self.scaler.mean_, self.scaler.scale_, self.kinds, self.groups, s)
        return self.Xs_tr[:, s], self.Xs_te[:, s], self._ti[n]


def outer_splits(y, repeats, n_splits=N_SPLITS, base_seed=SEED):
    """Repeat 0 = StratifiedKFold(5, shuffle, random_state=42) with seeds 42+fold — identical to the
    original notebooks. Repeat r>0 uses random_state=42+r and seeds 42+100r+fold."""
    out = []
    for r in range(repeats):
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=base_seed + r)
        for f, (tr, te) in enumerate(cv.split(np.zeros(len(y)), y), start=1):
            seed = base_seed + f if r == 0 else base_seed + 100 * r + f
            out.append((r, f, tr, te, seed))
    return out


def class_targets(y, alpha=1.0):
    cls, cnt = np.unique(y, return_counts=True)
    nmax = cnt.max()
    return {int(c): int(n + round(alpha * (nmax - n))) for c, n in zip(cls, cnt)}

In [ ]:
# ============================================================
# CELL 5 — Oversamplers
#   MDST and RA-MDST are the reconstructions from the submitted notebooks (same RNG call order).
#   RA-MDST-TP = RA-MDST with type-preserving synthesis (new, answers R1).
# ============================================================
def _snap(valid_sorted, x):
    i = np.searchsorted(valid_sorted, x)
    i = np.clip(i, 1, len(valid_sorted) - 1) if len(valid_sorted) > 1 else np.zeros_like(i)
    if len(valid_sorted) == 1:
        return valid_sorted[0]
    lo, hi = valid_sorted[i - 1], valid_sorted[i]
    return np.where(np.abs(x - lo) <= np.abs(hi - x), lo, hi)


def typed_interp(a, b, lam, rng, ti):
    """Type-preserving bounded pairwise interpolation.
    continuous: a + lam (b - a); ordinal: same, snapped to the nearest observed level;
    binary / label-encoded nominal: value copied from b with probability lam, else from a;
    mutually-exclusive binary groups: copied jointly from one parent (keeps one-hot validity)."""
    x = a + lam * (b - a)
    for j in ti.ord_idx:
        x[j] = _snap(ti.valid_s[j], x[j])
    take_b = rng.random(ti.p) < lam
    for g in ti.groups:
        take_b[g["cols"]] = take_b[g["cols"][0]]
    d = ti.disc_idx
    x[d] = np.where(take_b[d], b[d], a[d])
    return x


def mdst_family(X, y, targets, k=K_NEIGH, lam=LAM, seed=SEED, exclusive=False, bounded=False,
                typed=False, ti=None):
    """exclusive=False, bounded=False -> reconstructed MDST (alternating Euclidean/Manhattan centroids)
    exclusive=True,  bounded=True  -> RA-MDST;  + typed=True -> RA-MDST-TP
    exclusive=True,  bounded=False -> ablation 'Exclusive neighbours only'
    exclusive=False, bounded=True  -> ablation 'Bounded interpolation only' (6-position pool with duplicates)"""
    rng = np.random.default_rng(seed)
    X = np.asarray(X, float); y = np.asarray(y, int)
    X_new, y_new, par_s, par_q, overl = [], [], [], [], []
    for cls in np.unique(y):
        cls = int(cls)
        idx_c = np.where(y == cls)[0]
        need = int(targets.get(cls, len(idx_c)) - len(idx_c))
        if need <= 0 or len(idx_c) < 2:
            continue
        B = X[idx_c]
        de = pairwise_distances(B, metric="euclidean"); dm = pairwise_distances(B, metric="manhattan")
        np.fill_diagonal(de, np.inf); np.fill_diagonal(dm, np.inf)
        kk = min(k, len(B) - 1)
        cache = {}
        def neigh(s):
            if s not in cache:
                eu = [int(i) for i in np.argsort(de[s])[:kk]]
                man_orig = [int(i) for i in np.argsort(dm[s])[:kk]]
                man_x, eu_set = [], set(eu)
                for i in np.argsort(dm[s]):
                    i = int(i)
                    if i == s or i in eu_set:
                        continue
                    man_x.append(i)
                    if len(man_x) == kk:
                        break
                cache[s] = (eu, man_orig, man_x, len(eu_set & set(man_orig)) / kk)
            return cache[s]
        for t in range(need):
            s = int(rng.integers(0, len(B)))
            eu, man_orig, man_x, ov = neigh(s)
            man = man_x if exclusive else man_orig
            q = -1
            if bounded:
                pool = eu + man
                q = int(rng.choice(pool))
                l = float(rng.uniform(lam[0], lam[1]))
                syn = typed_interp(B[s].copy(), B[q], l, rng, ti) if typed else B[s] + l * (B[q] - B[s])
            else:
                mean_e = np.vstack([B[s], B[eu]]).mean(0); mean_m = np.vstack([B[s], B[man]]).mean(0)
                syn = mean_e if t % 2 == 0 else mean_m
            X_new.append(syn); y_new.append(cls)
            par_s.append(idx_c[s]); par_q.append(idx_c[q] if q >= 0 else -1); overl.append(ov)
    if not X_new:
        return X, y, {"parent_s": np.array([], int), "parent_q": np.array([], int), "overlap": np.array([])}
    return (np.vstack([X, np.asarray(X_new)]), np.concatenate([y, y_new]),
            {"parent_s": np.array(par_s), "parent_q": np.array(par_q), "overlap": np.array(overl)})


def oversample(name, X, y, ti=None, seed=SEED, targets=None, k=None, lam=LAM):
    """Unified interface. Returns X_aug, y_aug, meta. Synthetic rows are always appended after the real rows."""
    if targets is None:
        targets = class_targets(y, 1.0)
    meta = {"failed": False}
    if name == "None":
        return X, y, meta
    if name in ("MDST", "RA-MDST", "RA-MDST-TP", "Abl-ExclusiveOnly", "Abl-InterpOnly"):
        kk = K_NEIGH if k is None else k
        flags = {"MDST": (False, False, False), "RA-MDST": (True, True, False), "RA-MDST-TP": (True, True, True),
                 "Abl-ExclusiveOnly": (True, False, False), "Abl-InterpOnly": (False, True, False)}[name]
        Xa, ya, m = mdst_family(X, y, targets, k=kk, lam=lam, seed=seed, exclusive=flags[0],
                                bounded=flags[1], typed=flags[2], ti=ti)
        meta.update(m); return Xa, ya, meta
    strat = {c: t for c, t in targets.items() if t > (y == c).sum()}
    if not strat:
        return X, y, meta
    min_n = min(int((y == c).sum()) for c in strat)
    kk = IMB_K if k is None else k
    kn = max(1, min(kk, min_n - 1))
    try:
        if name == "ROS":
            s = RandomOverSampler(sampling_strategy=strat, random_state=seed)
        elif name == "SMOTE":
            s = SMOTE(sampling_strategy=strat, k_neighbors=kn, random_state=seed)
        elif name == "Borderline-SMOTE":
            s = BorderlineSMOTE(sampling_strategy=strat, k_neighbors=kn, m_neighbors=10, random_state=seed)
        elif name == "ADASYN":
            s = ADASYN(sampling_strategy=strat, n_neighbors=kn, random_state=seed)
        elif name == "SMOTE-NC":
            cat = list(ti.cat_idx) if ti is not None else []
            if len(cat) == 0:
                s = SMOTE(sampling_strategy=strat, k_neighbors=kn, random_state=seed)
            elif len(cat) == X.shape[1]:
                s = SMOTEN(sampling_strategy=strat, k_neighbors=kn, random_state=seed)
            else:
                s = SMOTENC(categorical_features=cat, sampling_strategy=strat, k_neighbors=kn, random_state=seed)
        else:
            raise ValueError(name)
        Xa, ya = s.fit_resample(X, y)
        return np.asarray(Xa, float), np.asarray(ya, int), meta
    except Exception as e:   # e.g. Borderline-SMOTE finds no 'danger' samples, ADASYN finds no majority neighbours
        meta["failed"] = True; meta["error"] = str(e)[:200]
        return X, y, meta

In [ ]:
# ============================================================
# CELL 6 — Classifiers, metrics, validity and diversity measures
# ============================================================
def get_model(name, seed=SEED, C=C_SVM, max_iter=None):
    mi = SVC_MAX_ITER if max_iter is None else max_iter
    if name == "LogisticRegression": return LogisticRegression(C=1.0, max_iter=5000, random_state=seed)
    if name == "LinearSVM": return SVC(kernel="linear", C=C, random_state=seed, max_iter=mi, cache_size=500)
    if name == "RBFSVM": return SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed, max_iter=mi, cache_size=500)
    if name == "KNN": return KNeighborsClassifier(n_neighbors=5, weights="distance")
    if name == "DecisionTree": return DecisionTreeClassifier(random_state=seed)
    if name == "RandomForest": return RandomForestClassifier(n_estimators=TREE_EST, random_state=seed, n_jobs=1)
    if name == "ExtraTrees": return ExtraTreesClassifier(n_estimators=TREE_EST, random_state=seed, n_jobs=1)
    if name == "GradientBoosting": return GradientBoostingClassifier(n_estimators=GB_EST, learning_rate=0.05, random_state=seed)
    if name == "AdaBoost": return AdaBoostClassifier(n_estimators=GB_EST, learning_rate=0.05, random_state=seed)
    if name == "MLP": return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-4, learning_rate_init=1e-3,
                                           max_iter=MLP_ITER, early_stopping=True, validation_fraction=0.15,
                                           n_iter_no_change=30, random_state=seed)
    raise ValueError(name)


def ordinal_codes(yv):
    return np.array([ORD_MAP[int(v)] for v in yv])


def metrics(y_true, y_pred):
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
         "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
         "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
         "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
         "mcc": matthews_corrcoef(y_true, y_pred)}
    R = recall_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    F = f1_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    for i, c in enumerate(CLASSES):
        m[f"recall_{CLASS_NAMES[c]}"] = R[i]; m[f"f1_{CLASS_NAMES[c]}"] = F[i]
    if ORD_MAP is not None:
        ot, op = ordinal_codes(y_true), ordinal_codes(y_pred)
        m["ord_mae"] = float(np.mean(np.abs(ot - op)))
        m["ord_qwk"] = cohen_kappa_score(ot, op, weights="quadratic")
    return m


def validity(X_syn, ti):
    """Share of synthetic values that cannot occur in the real data (computed on the original scale)."""
    if X_syn is None or len(X_syn) == 0:
        return {}
    raw = X_syn * ti.scale + ti.mean
    bad = np.zeros_like(raw, dtype=bool)
    for j in range(ti.p):
        kind, v = ti.kinds[j], raw[:, j]
        if kind == "binary":
            bad[:, j] = ~((np.abs(v) < 1e-6) | (np.abs(v - 1) < 1e-6))
        elif kind in ("nominal", "ordinal"):
            vr = ti.valid_raw[j]
            idx = np.clip(np.searchsorted(vr, v), 0, len(vr) - 1)
            idx2 = np.clip(idx - 1, 0, len(vr) - 1)
            bad[:, j] = np.minimum(np.abs(vr[idx] - v), np.abs(vr[idx2] - v)) > 1e-6
        else:
            bad[:, j] = (v < ti.lo[j] - 1e-9) | (v > ti.hi[j] + 1e-9)
    out = {"pct_invalid_values": 100 * bad.mean(),
           "pct_rows_with_invalid": 100 * bad.any(1).mean()}
    for kind in ["binary", "nominal", "ordinal", "continuous"]:
        cols = np.where(ti.kinds == kind)[0]
        out[f"pct_invalid_{kind}"] = 100 * bad[:, cols].mean() if len(cols) else np.nan
    if ti.groups:
        viol = np.zeros(len(raw), bool)
        for g in ti.groups:
            rb = np.round(raw[:, g["cols"]], 6)
            okb = np.isin(rb, [0.0, 1.0]).all(1)
            s = rb.sum(1)
            viol |= (~okb) | ((s != 1) if g["exact"] else (s > 1))
        out["pct_rows_group_violation"] = 100 * viol.mean()
    else:
        out["pct_rows_group_violation"] = np.nan
    return out


def diversity(X_real, y_real, X_syn, y_syn, k_cov=5, near_frac=0.1):
    """Direct diversity / fidelity / separability measures for synthetic samples (scaled space).
    Coverage & density follow Naeem et al. (2020)."""
    rows = []
    for c in np.unique(y_syn):
        R = X_real[y_real == c]; S = X_syn[y_syn == c]
        if len(S) == 0 or len(R) < k_cov + 1:
            continue
        dRR = cdist(R, R); np.fill_diagonal(dRR, np.inf)
        nnR = dRR.min(1); med = np.median(nnR); tau = near_frac * med
        dSR = cdist(S, R); nnSR = dSR.min(1)
        rk = np.sort(dRR, 1)[:, k_cov - 1]                       # k-NN radius of each real point
        inside = dSR <= rk[None, :]                                # syn j inside ball of real i
        coverage = inside.any(0).mean()
        density = inside.sum() / (k_cov * len(S))
        if len(S) > 1:
            dSS = cdist(S, S); iu = np.triu_indices(len(S), 1)
            mean_pair_S = dSS[iu].mean(); np.fill_diagonal(dSS, np.inf)
            near_dup_syn = (dSS.min(1) < tau).mean()
        else:
            mean_pair_S, near_dup_syn = np.nan, np.nan
        mean_pair_R = dRR[np.isfinite(dRR)].mean()
        uniq = len(np.unique(np.round(S, 8), axis=0)) / len(S)
        dAll = cdist(S, X_real)
        nn_all = np.argsort(dAll, 1)[:, :5]
        intrusion = (y_real[nn_all[:, 0]] != c).mean()
        purity5 = (y_real[nn_all] == c).mean()
        rows.append({"class": CLASS_NAMES.get(int(c), int(c)), "n_syn": len(S),
                     "unique_ratio": uniq, "exact_dup_of_real": (nnSR < 1e-8).mean(),
                     "near_dup_of_real": (nnSR < tau).mean(), "near_dup_within_syn": near_dup_syn,
                     "rel_nn_dist_to_real": nnSR.mean() / med, "rel_pairwise_spread": mean_pair_S / mean_pair_R,
                     "coverage": coverage, "density": density,
                     "intrusion_rate": intrusion, "knn5_purity": purity5})
    return rows

## Section 1 — Neighborhood-overlap audit (Table 3, Fig. 3/7, R1 "0.7582 vs 0.7554", R2 #5)
The overlap is computed three ways so the manuscript can state exactly which one each number is:
**(a)** every observation of every class in the full scaled + RFE-100 space (the Table 3 definition),
**(b)** minority-class observations only, and **(c)** per *generated* sample inside the Protocol B training folds
(seeds drawn with replacement) — this is what `main (1).ipynb` printed as 0.7554.

In [ ]:
def section1():
    sc = StandardScaler().fit(X_RAW)
    Xs = sc.transform(X_RAW)
    sel = rfe_support(Xs, Y, N_RFE, SEED)
    Xf = Xs[:, sel]
    rows = []
    for c in CLASSES:
        idx = np.where(Y == c)[0]; B = Xf[idx]
        de = pairwise_distances(B); dm = pairwise_distances(B, metric="manhattan")
        np.fill_diagonal(de, np.inf); np.fill_diagonal(dm, np.inf)
        kk = min(K_NEIGH, len(B) - 1)
        for i in range(len(B)):
            e = set(np.argsort(de[i])[:kk].tolist()); m = set(np.argsort(dm[i])[:kk].tolist())
            rows.append({"class": CLASS_NAMES[c], "shared": len(e & m), "overlap": len(e & m) / kk})
    ov = pd.DataFrame(rows)
    t3 = ov.groupby("shared").size().rename("n_observations").reset_index()
    t3["percent"] = 100 * t3["n_observations"] / len(ov)
    save_table(t3, "T01_overlap_distribution_all_observations.csv")
    by_cls = ov.groupby("class")["overlap"].agg(["mean", "count"]).reset_index()
    save_table(by_cls, "T01b_overlap_by_class.csv")
    a = ov["overlap"].mean()
    b = ov[ov["class"] != CLASS_NAMES[CLASSES[np.argmax(np.bincount(Y))]]]["overlap"].mean()
    # (c) per generated sample inside Protocol B folds (repeat 0)
    gen = []
    for r, f, tr, te, seed in outer_splits(Y, 1):
        ctx = FoldCtx(X_RAW, Y, tr, te, seed)
        Xtr, _, ti = ctx.data(N_RFE)
        _, _, meta = oversample("RA-MDST", Xtr, ctx.y_tr, ti, seed)
        gen.append(meta["overlap"])
    c_ = float(np.concatenate(gen).mean())
    print(t3.to_string(index=False))
    print(f"(a) all {len(ov)} observations: mean overlap = {a:.4f}")
    print(f"(b) minority observations only : mean overlap = {b:.4f}")
    print(f"(c) per generated sample, Protocol B folds (old notebook value 0.7554): {c_:.4f}")
    SUMMARY["overlap"] = {"all_obs": a, "minority_obs": b, "per_generated_sample_protocolB": c_,
                          "distribution": t3.to_dict("records")}
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.bar(t3["shared"].astype(str), t3["percent"], color="#2a78d6", width=0.6)
    for x_, p_ in zip(t3["shared"].astype(str), t3["percent"]):
        ax.text(x_, p_ + 1, f"{p_:.1f}%", ha="center", fontsize=10)
    ax.set_xlabel(f"Shared neighbours between Euclidean and Manhattan top-{K_NEIGH}")
    ax.set_ylabel("Observations (%)")
    ax.set_title(f"Mean overlap = {a:.4f} (n = {len(ov)})", fontsize=11, loc="left")
    savefig(fig, "F01_overlap_distribution")
    return ov

OVERLAP = run_section("Section 1 — overlap audit", section1)

## Section 2 — Protocol A reproduction and leakage diagnostics
Protocol A = the predecessor workflow from `main (2).ipynb`: scaling + RFE on all 324 rows → oversample to 711 → 10-fold CV **on the augmented set**.
Seed 42 reproduces the submitted Protocol A table. Other seeds show how much the headline number moves with the random seed
(this explains 93.11 vs 92.41 vs 92.76). The diagnostics show *why* Protocol A is optimistic: test folds contain synthetic
rows whose real parents are in the training folds.

In [ ]:
def section2():
    sc = StandardScaler().fit(X_RAW); Xs = sc.transform(X_RAW)
    sel = rfe_support(Xs, Y, N_RFE, SEED)
    Xf = Xs[:, sel]
    ti = TypeInfo(X_RAW, sc.mean_, sc.scale_, KINDS_ALL, GROUPS_ALL, sel)
    n = len(Y)
    rows = []
    samplers = SAMPLERS_MAIN + ABLATION_SAMPLERS
    for seed in PROTO_A_SEEDS:
        for s_name in (samplers if seed == PROTO_A_SEEDS[0] else ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]):
            Xa, ya, meta = oversample(s_name, Xf, Y, ti, seed, targets=class_targets(Y, 1.0))
            is_syn = np.r_[np.zeros(n, bool), np.ones(len(ya) - n, bool)]
            ps = meta.get("parent_s", np.array([], int))
            cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
            for fold, (tr, te) in enumerate(cv.split(Xa, ya), start=1):
                clf = SVC(kernel="linear", C=C_SVM, random_state=seed + fold, max_iter=SVC_MAX_ITER).fit(Xa[tr], ya[tr])
                pred = clf.predict(Xa[te])
                m = metrics(ya[te], pred)
                te_syn = te[is_syn[te]]; te_real = te[~is_syn[te]]
                m.update({"seed": seed, "sampler": s_name, "fold": fold, "n_test": len(te),
                          "pct_test_synthetic": 100 * len(te_syn) / len(te),
                          "acc_real_test_rows": accuracy_score(ya[te_real], pred[~is_syn[te]]) if len(te_real) else np.nan,
                          "acc_synthetic_test_rows": accuracy_score(ya[te_syn], pred[is_syn[te]]) if len(te_syn) else np.nan,
                          "f1_real_test_rows": f1_score(ya[te_real], pred[~is_syn[te]], average="macro") if len(te_real) else np.nan})
                if len(te_syn) and len(ps):
                    tr_real = set(tr[~is_syn[tr]].tolist())
                    par = ps[te_syn - n]
                    m["pct_syn_test_parent_in_train"] = 100 * np.mean([p in tr_real for p in par])
                    dmin = cdist(Xa[te_syn], Xa[tr]).min(1)
                    m["pct_syn_test_exact_copy_in_train"] = 100 * np.mean(dmin < 1e-8)
                elif len(te_syn):
                    dmin = cdist(Xa[te_syn], Xa[tr]).min(1)
                    m["pct_syn_test_exact_copy_in_train"] = 100 * np.mean(dmin < 1e-8)
                rows.append(m)
    A = pd.DataFrame(rows)
    save_table(A, "T02_protocolA_fold_results_all_seeds.csv")
    agg_cols = ["accuracy", "f1_macro", "balanced_accuracy", "acc_real_test_rows", "acc_synthetic_test_rows",
                "f1_real_test_rows", "pct_test_synthetic", "pct_syn_test_parent_in_train", "pct_syn_test_exact_copy_in_train"]
    agg_cols = [c for c in agg_cols if c in A.columns]
    s42 = A[A.seed == PROTO_A_SEEDS[0]].groupby("sampler", sort=False)[agg_cols].mean().reset_index()
    for c in ["accuracy", "f1_macro", "balanced_accuracy", "acc_real_test_rows", "acc_synthetic_test_rows", "f1_real_test_rows"]:
        s42[c] *= 100
    save_table(s42.round(2), "T02a_protocolA_seed42_reproduction.csv")
    print("Protocol A, seed", PROTO_A_SEEDS[0], "(should match main (2).ipynb: MDST 90.58 / RA-MDST 92.41 accuracy)")
    print(s42.round(2).to_string(index=False))
    sweep = (A.groupby(["sampler", "seed"])[["accuracy", "f1_macro", "balanced_accuracy"]].mean() * 100).reset_index()
    sw = sweep.groupby("sampler")[["accuracy", "f1_macro"]].agg(["mean", "std", "min", "max"]).round(2)
    sw.columns = ["_".join(c) for c in sw.columns]
    save_table(sw.reset_index(), "T02b_protocolA_seed_sweep.csv")
    print("\nSeed sweep (accuracy / macro-F1 across seeds):\n", sw.to_string())
    SUMMARY["protocolA_seed42"] = s42.round(2).to_dict("records")
    SUMMARY["protocolA_seed_sweep"] = sw.reset_index().to_dict("records")
    # Corrected Table 2 (real counts only)
    rows2 = []
    for r, f, tr, te, seed in outer_splits(Y, 1):
        for c in CLASSES:
            rows2.append({"fold": f, "class": CLASS_NAMES[c], "train_real": int((Y[tr] == c).sum()),
                          "test_real": int((Y[te] == c).sum()),
                          "train_after_balancing": int(class_targets(Y[tr])[int(c)])})
    t2 = pd.DataFrame(rows2)
    save_table(t2, "T02c_protocolB_fold_class_counts.csv")
    # figure: seed sweep
    fig, ax = plt.subplots(figsize=(7, 3.6))
    order = ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]
    for i, s_name in enumerate(order):
        v = sweep[sweep.sampler == s_name]["accuracy"].values
        ax.scatter(np.full(len(v), i) + np.random.default_rng(0).uniform(-0.12, 0.12, len(v)), v,
                   s=28, color=SAMPLER_COLORS[s_name], alpha=0.85)
        ax.hlines(v.mean(), i - 0.25, i + 0.25, color="#0b0b0b", lw=2)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
    ax.set_ylabel("Protocol A accuracy (%)")
    ax.set_title("Protocol A accuracy across random seeds (each dot = one seed)", loc="left", fontsize=11)
    savefig(fig, "F02_protocolA_seed_sweep")
    return A

PROTO_A = run_section("Section 2 — Protocol A reproduction", section2)

## Section 3 — Protocol B benchmark (leakage-controlled)
For every outer fold: **split → scaler fit on real training rows → RFE-100 fit on real training rows →
oversampling of the training rows only → classifier → evaluation on the untouched real test fold.**
Design: `MAIN_REPEATS` × 5-fold stratified CV (repeat 0 is identical to the submitted Protocol B), 9 samplers × 10 classifiers,
plus the two ablation variants with 3 classifiers. Every sampler balances each minority class to the majority count of that training fold.

In [ ]:
def main_fold_task(r, f, tr, te, seed):
    ctx = FoldCtx(X_RAW, Y, tr, te, seed)
    Xtr, Xte, ti = ctx.data(N_RFE)
    ytr, yte = ctx.y_tr, ctx.y_te
    perf, val, div, preds = [], [], [], []
    for s_name in SAMPLERS_MAIN + ABLATION_SAMPLERS:
        t0 = time.perf_counter()
        Xa, ya, meta = oversample(s_name, Xtr, ytr, ti, seed)
        t_s = time.perf_counter() - t0
        n_syn = len(ya) - len(ytr)
        base = {"repeat": r, "fold": f, "seed": seed, "sampler": s_name}
        if n_syn > 0:
            v = validity(Xa[len(ytr):], ti); v.update(base); v["n_syn"] = n_syn; val.append(v)
            for d in diversity(Xtr, ytr, Xa[len(ytr):], ya[len(ytr):]):
                d.update(base); div.append(d)
            try:
                sil = silhouette_score(Xa, ya)
            except Exception:
                sil = np.nan
            div.append({**base, "class": "ALL", "silhouette_augmented": sil})
        clfs = CLASSIFIERS if s_name in SAMPLERS_MAIN else ABLATION_CLASSIFIERS
        for c_name in clfs:
            model = get_model(c_name, seed)
            t1 = time.perf_counter(); model.fit(Xa, ya); t_fit = time.perf_counter() - t1
            pred = model.predict(Xte)
            m = metrics(yte, pred)
            m.update(base); m.update({"classifier": c_name, "n_train_real": len(ytr), "n_train_aug": len(ya),
                                      "n_syn": n_syn, "sampling_sec": t_s, "fit_sec": t_fit,
                                      "sampler_failed": meta.get("failed", False)})
            perf.append(m)
            preds.append(pd.DataFrame({**base, "classifier": c_name, "idx": te, "y_true": yte, "y_pred": pred}))
    return perf, val, div, pd.concat(preds)


def section3():
    tasks = outer_splits(Y, MAIN_REPEATS)
    res = Parallel(n_jobs=N_JOBS, verbose=5, timeout=TASK_TIMEOUT_SEC)(delayed(main_fold_task)(*t) for t in tasks)
    perf = pd.DataFrame([x for r_ in res for x in r_[0]])
    val = pd.DataFrame([x for r_ in res for x in r_[1]])
    div = pd.DataFrame([x for r_ in res for x in r_[2]])
    preds = pd.concat([r_[3] for r_ in res], ignore_index=True)
    save_table(perf, "T03_protocolB_fold_results.csv")
    save_table(preds, "T03_protocolB_oof_predictions.csv")
    save_table(val, "T06_validity_fold_level.csv")
    save_table(div, "T07_diversity_fold_level.csv")
    mcols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "balanced_accuracy", "f1_weighted", "mcc",
             "ord_mae", "ord_qwk"] + [c for c in perf.columns if c.startswith("recall_") and c != "recall_macro"]
    g = perf.groupby(["classifier", "sampler"])[mcols]
    summ = g.mean().add_suffix("_mean").join(g.std().add_suffix("_sd")).reset_index()
    save_table(summ, "T03a_protocolB_summary_all.csv")
    piv = (perf.groupby(["classifier", "sampler"])["f1_macro"].mean().unstack() * 100)
    piv = piv[[s for s in SAMPLERS_MAIN + ABLATION_SAMPLERS if s in piv.columns]].loc[[c for c in CLASSIFIERS if c in piv.index]]
    save_table(piv.round(2), "T03b_protocolB_macroF1_matrix.csv", index=True)
    # Linear SVM table (the paper's main model) — mean ± sd over all folds
    ls = perf[perf.classifier == "LinearSVM"]
    rows = []
    for s_name in SAMPLERS_MAIN + ABLATION_SAMPLERS:
        d = ls[ls.sampler == s_name]
        if not len(d):
            continue
        row = {"sampler": s_name}
        for mtr in ["accuracy", "f1_macro", "balanced_accuracy", "precision_macro", "recall_macro", "mcc", "ord_qwk"]:
            row[mtr] = f"{100*d[mtr].mean():.2f} ± {100*d[mtr].std():.2f}" if mtr not in ("mcc", "ord_qwk") else f"{d[mtr].mean():.3f} ± {d[mtr].std():.3f}"
        r0 = d[d.repeat == 0]
        row["repeat0_acc"] = round(100 * r0.accuracy.mean(), 2); row["repeat0_f1"] = round(100 * r0.f1_macro.mean(), 2)
        row["failed_folds"] = int(d.sampler_failed.sum())
        rows.append(row)
    lst = pd.DataFrame(rows)
    save_table(lst, "T03c_protocolB_LinearSVM_table.csv")
    print("Linear SVM, Protocol B (mean ± sd over", MAIN_REPEATS * N_SPLITS, "folds). 'repeat0_*' = the single 5-fold run of the submission")
    print(lst.to_string(index=False))
    print("\nMacro-F1 (%) — classifier × sampler")
    print(piv.round(2).to_string())
    SUMMARY["protocolB_linearSVM"] = lst.to_dict("records")
    SUMMARY["protocolB_macroF1_matrix"] = piv.round(2).reset_index().to_dict("records")
    # Figure: heatmap
    fig, ax = plt.subplots(figsize=(11, 5))
    im = ax.imshow(piv.values, cmap="Blues", aspect="auto")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=35, ha="right")
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
    ax.grid(False)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5,
                        color="white" if v > np.nanpercentile(piv.values, 70) else "#0b0b0b")
    fig.colorbar(im, ax=ax, label="Macro-F1 (%)", shrink=0.8)
    ax.set_title(f"Protocol B macro-F1, mean of {MAIN_REPEATS}×{N_SPLITS} folds", loc="left", fontsize=11)
    savefig(fig, "F03_protocolB_macroF1_heatmap")
    # Figure: Protocol A vs B for Linear SVM
    if PROTO_A is not None:
        a = PROTO_A[PROTO_A.seed == PROTO_A_SEEDS[0]].groupby("sampler")[["accuracy", "f1_macro"]].mean() * 100
        b = ls.groupby("sampler")[["accuracy", "f1_macro"]].mean() * 100
        order = [s for s in SAMPLERS_MAIN if s in a.index and s in b.index]
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
        for ax, mtr, lab in zip(axes, ["accuracy", "f1_macro"], ["Accuracy (%)", "Macro-F1 (%)"]):
            x = np.arange(len(order))
            ax.bar(x - 0.2, a.loc[order, mtr], 0.38, color="#b7d3f6", label="Protocol A (leaky)")
            ax.bar(x + 0.2, b.loc[order, mtr], 0.38, color="#184f95", label="Protocol B (leakage-controlled)")
            ax.set_xticks(x); ax.set_xticklabels(order, rotation=35, ha="right"); ax.set_ylabel(lab)
        h, l = axes[0].get_legend_handles_labels()
        fig.legend(h, l, loc="upper right", ncol=2, bbox_to_anchor=(0.98, 1.02))
        fig.suptitle("Linear SVM: the same samplers under Protocol A vs Protocol B", x=0.06, y=1.0, ha="left", fontsize=11)
        savefig(fig, "F03b_protocolA_vs_B_linearSVM")
    # Confusion matrices (Linear SVM, all repeats pooled, row-normalised)
    show = ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]
    fig, axes = plt.subplots(1, len(show), figsize=(3.3 * len(show), 3.4))
    for ax, s_name in zip(axes, show):
        p = preds[(preds.sampler == s_name) & (preds.classifier == "LinearSVM")]
        cm = confusion_matrix(p.y_true, p.y_pred, labels=CLASSES)
        cmn = cm / cm.sum(1, keepdims=True)
        ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues"); ax.grid(False)
        for i in range(len(CLASSES)):
            for j in range(len(CLASSES)):
                ax.text(j, i, f"{100*cmn[i,j]:.0f}%", ha="center", va="center", fontsize=9,
                        color="white" if cmn[i, j] > 0.6 else "#0b0b0b")
        ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels([CLASS_NAMES[c] for c in CLASSES], fontsize=9)
        ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels([CLASS_NAMES[c] for c in CLASSES], fontsize=9)
        ax.set_title(s_name, fontsize=10); ax.set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    savefig(fig, "F03c_confusion_linearSVM")
    return perf, val, div, preds

MAIN = run_section("Section 3 — Protocol B benchmark", section3)

## Section 4 — Statistics
For each pair of samplers and each classifier, using the paired fold-level differences:
* mean Δ with the **Nadeau–Bengio corrected resampled t-test** and its corrected 95% CI (accounts for overlapping training sets in repeated CV);
* two-sided **Wilcoxon signed-rank** on the `MAIN_REPEATS` repeat-level means, with the **matched-pairs rank-biserial** effect size;
* **cluster-bootstrap 95% CI** of Δ (resampling repeats);
* **Holm** and **Benjamini–Hochberg** correction across the 10 classifiers;
* **Friedman test** and average ranks of all samplers across classifiers (Demšar 2006).
The table also records the smallest p-value a one-sided Wilcoxon can reach with only 5 folds (1/32 = 0.03125), which is the limitation R1 raised.

In [ ]:
def holm(p):
    p = np.asarray(p, float); n = np.sum(np.isfinite(p)); out = np.full_like(p, np.nan)
    order = np.argsort(np.where(np.isfinite(p), p, np.inf)); running = 0
    for rank, i in enumerate(order):
        if not np.isfinite(p[i]): continue
        running = max(running, min(1, (n - rank) * p[i])); out[i] = running
    return out

def bh(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    pv = p[ok]; n = len(pv)
    if n == 0: return out
    o = np.argsort(pv); adj = pv[o] * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]; res = np.empty(n); res[o] = np.minimum(adj, 1)
    out[ok] = res; return out

def nb_corrected(d, n_train, n_test):
    d = np.asarray(d, float); J = len(d); mu = d.mean(); var = d.var(ddof=1)
    se = math.sqrt((1 / J + n_test / n_train) * var) if var > 0 else 0.0
    if se == 0:
        return mu, (1.0 if mu == 0 else 0.0), mu, mu
    t = mu / se; p = 2 * stats.t.sf(abs(t), J - 1); h = stats.t.ppf(0.975, J - 1) * se
    return mu, p, mu - h, mu + h

def rank_biserial(d):
    d = np.asarray(d, float); d = d[d != 0]
    if len(d) == 0: return 0.0
    r = stats.rankdata(np.abs(d)); return (r[d > 0].sum() - r[d < 0].sum()) / r.sum()

COMPARISONS = [("RA-MDST", "MDST"), ("RA-MDST-TP", "MDST"), ("RA-MDST-TP", "RA-MDST"),
               ("RA-MDST", "SMOTE"), ("RA-MDST-TP", "SMOTE"), ("RA-MDST", "None"), ("RA-MDST-TP", "None"),
               ("MDST", "None"), ("SMOTE", "None"), ("RA-MDST", "ROS")]

def section4():
    perf = MAIN[0]
    rng = np.random.default_rng(SEED)
    rows = []
    n_test = len(Y) / N_SPLITS; n_train = len(Y) - n_test
    for a, b in COMPARISONS:
        for mtr in ["f1_macro", "balanced_accuracy", "accuracy"]:
            block = []
            for c_name in CLASSIFIERS:
                A = perf[(perf.sampler == a) & (perf.classifier == c_name)].set_index(["repeat", "fold"])[mtr]
                B = perf[(perf.sampler == b) & (perf.classifier == c_name)].set_index(["repeat", "fold"])[mtr]
                d = (A - B).dropna()
                if len(d) < 2: continue
                mu, p_nb, lo, hi = nb_corrected(d.values, n_train, n_test)
                rep = d.groupby(level=0).mean().values
                try:
                    p_w = stats.wilcoxon(rep, alternative="two-sided").pvalue if np.any(rep != 0) else 1.0
                except Exception:
                    p_w = np.nan
                boots = [rng.choice(rep, len(rep), replace=True).mean() for _ in range(5000)] if len(rep) > 1 else [mu]
                block.append({"comparison": f"{a} − {b}", "metric": mtr, "classifier": c_name,
                              "mean_A": 100 * A.mean(), "mean_B": 100 * B.mean(), "delta_pp": 100 * mu,
                              "nb_ci_low_pp": 100 * lo, "nb_ci_high_pp": 100 * hi, "p_nadeau_bengio": p_nb,
                              "boot_ci_low_pp": 100 * np.percentile(boots, 2.5), "boot_ci_high_pp": 100 * np.percentile(boots, 97.5),
                              "p_wilcoxon_repeats": p_w, "rank_biserial_r": rank_biserial(rep),
                              "cohens_dz_folds": mu / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan,
                              "wins_folds": int((d > 0).sum()), "ties_folds": int((d == 0).sum()), "losses_folds": int((d < 0).sum()),
                              "n_folds": len(d), "n_repeats": len(rep)})
            if not block: continue
            bdf = pd.DataFrame(block)
            bdf["p_nb_holm"] = holm(bdf.p_nadeau_bengio); bdf["p_nb_bh"] = bh(bdf.p_nadeau_bengio)
            bdf["p_wilcoxon_holm"] = holm(bdf.p_wilcoxon_repeats)
            rows.append(bdf)
    st = pd.concat(rows, ignore_index=True)
    # Primary family for Table 9: Holm across all 60 tests (6 comparisons x 10 classifiers), per metric.
    # p_nb_holm (above) is the more liberal per-comparison family (10 classifiers).
    TABLE9 = ["RA-MDST − MDST", "RA-MDST-TP − MDST", "RA-MDST-TP − SMOTE", "RA-MDST − None", "RA-MDST-TP − None", "MDST − None"]
    st["p_nb_holm_table9_60tests"] = np.nan
    for mtr in st.metric.unique():
        mk = (st.metric == mtr) & st.comparison.isin(TABLE9)
        st.loc[mk, "p_nb_holm_table9_60tests"] = holm(st.loc[mk, "p_nadeau_bengio"].values)
        mk_all = st.metric == mtr
        st.loc[mk_all, "p_nb_holm_all_100tests"] = holm(st.loc[mk_all, "p_nadeau_bengio"].values)
    st["min_attainable_p_wilcoxon_5folds_one_sided"] = 1 / 32
    save_table(st.round(5), "T04_pairwise_statistics.csv")
    KEY_COMPS = ["RA-MDST − MDST", "RA-MDST-TP − MDST", "RA-MDST-TP − SMOTE",
                 "RA-MDST − None", "RA-MDST-TP − None", "MDST − None", "SMOTE − None"]
    key = st[(st.metric == "f1_macro") & st.comparison.isin(KEY_COMPS)]
    print(key[["comparison", "classifier", "delta_pp", "nb_ci_low_pp", "nb_ci_high_pp", "p_nadeau_bengio", "p_nb_holm",
               "p_wilcoxon_repeats", "p_wilcoxon_holm", "rank_biserial_r"]].round(4).to_string(index=False))
    SUMMARY["stats_key"] = key[["comparison", "classifier", "delta_pp", "nb_ci_low_pp", "nb_ci_high_pp",
                                "p_nb_holm", "p_wilcoxon_holm", "rank_biserial_r", "cohens_dz_folds"]].round(4).to_dict("records")
    # Which comparisons survive Holm correction (Nadeau-Bengio test), per metric
    _p = st.p_nb_holm_table9_60tests.fillna(st.p_nb_holm)   # 60-test family for Table 9 comparisons
    surv = (st.assign(sig=_p < 0.05, pos=(_p < 0.05) & (st.delta_pp > 0), neg=(_p < 0.05) & (st.delta_pp < 0),
                      sig_within_comparison=st.p_nb_holm < 0.05)
              .groupby(["metric", "comparison"]).agg(n_classifiers=("classifier", "size"), n_significant=("sig", "sum"),
                                                     n_sig_better=("pos", "sum"), n_sig_worse=("neg", "sum"),
                                                     n_sig_within_comparison=("sig_within_comparison", "sum"),
                                                     significant_for=("classifier", lambda c: ", ".join(
                                                         c[_p.loc[c.index] < 0.05])))
              .reset_index())
    save_table(surv, "T04c_holm_survival_summary.csv")
    print("\nComparisons surviving Holm correction (Nadeau-Bengio, alpha = 0.05):")
    print(surv[surv.metric == "f1_macro"].to_string(index=False))
    SUMMARY["stats_survival"] = surv[surv.metric == "f1_macro"].to_dict("records")
    # Friedman + average ranks across classifiers (blocks)
    fr_rows = []
    for mtr in ["f1_macro", "balanced_accuracy", "accuracy"]:
        M = perf[perf.sampler.isin(SAMPLERS_MAIN)].groupby(["classifier", "sampler"])[mtr].mean().unstack()[SAMPLERS_MAIN]
        ranks = M.rank(axis=1, ascending=False).mean()
        try:
            fs = stats.friedmanchisquare(*[M[c].values for c in M.columns])
            p_f = fs.pvalue; chi = fs.statistic
        except Exception:
            p_f = chi = np.nan
        kq = len(SAMPLERS_MAIN); Nb = len(M)
        q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}.get(kq, np.nan)
        cd = q05 * math.sqrt(kq * (kq + 1) / (6 * Nb))
        for s_name, rk in ranks.items():
            fr_rows.append({"metric": mtr, "sampler": s_name, "avg_rank": rk, "friedman_chi2": chi, "friedman_p": p_f,
                            "nemenyi_CD_0.05": cd, "n_classifiers": Nb})
        if mtr == "f1_macro":
            rs = ranks.sort_values()
            fig, ax = plt.subplots(figsize=(7, 3.4))
            ax.barh(rs.index[::-1], rs.values[::-1], color=[SAMPLER_COLORS[s] for s in rs.index[::-1]], height=0.6)
            ax.set_xlabel("Average rank across 10 classifiers (lower = better)")
            ax.errorbar([rs.min()], [len(rs) - 1], xerr=[[0], [cd]], color="#0b0b0b", capsize=4, lw=1.5)
            ax.text(rs.min() + cd / 2, len(rs) - 0.55, f"CD = {cd:.2f}", ha="center", fontsize=9)
            ax.set_title(f"Macro-F1 ranks — Friedman p = {p_f:.3g}", loc="left", fontsize=11)
            savefig(fig, "F04b_average_ranks")
    fr = pd.DataFrame(fr_rows)
    save_table(fr.round(4), "T04b_friedman_average_ranks.csv")
    SUMMARY["friedman"] = fr[fr.metric == "f1_macro"].round(4).to_dict("records")
    # Forest plot of macro-F1 deltas with corrected CIs
    comps = ["RA-MDST − MDST", "RA-MDST-TP − MDST", "RA-MDST-TP − SMOTE", "RA-MDST − None", "RA-MDST-TP − None"]
    fig, axes = plt.subplots(1, len(comps), figsize=(3.6 * len(comps), 4.4), sharey=True)
    for ax, comp in zip(axes, comps):
        d = st[(st.comparison == comp) & (st.metric == "f1_macro")].set_index("classifier").loc[CLASSIFIERS]
        yy = np.arange(len(d))[::-1]
        col = SAMPLER_COLORS[comp.split(" − ")[0]]
        ax.hlines(yy, d.nb_ci_low_pp, d.nb_ci_high_pp, color=col, lw=2)
        ax.scatter(d.delta_pp, yy, color=col, s=36, zorder=3)
        ax.axvline(0, color="#52514e", lw=1)
        for yv, (_, rr) in zip(yy, d.iterrows()):
            if rr.p_nb_holm < 0.05:
                ax.text(rr.nb_ci_high_pp, yv, "  *", va="center", fontsize=12)
        ax.set_yticks(yy); ax.set_yticklabels(d.index)
        ax.set_xlabel("Δ macro-F1 (pp), corrected 95% CI"); ax.set_title(comp, fontsize=11)
    savefig(fig, "F04_delta_forest_macroF1")
    return st

STATS = run_section("Section 4 — statistics", section4) if MAIN is not None else None

## Section 5 — Ablation under Protocol B
Same folds as Section 3. `MDST` = original overlapping neighbourhoods + centroid rule; `Abl-ExclusiveOnly` = exclusive Manhattan neighbours + centroid rule;
`Abl-InterpOnly` = overlapping neighbourhoods + bounded pairwise interpolation; `RA-MDST` = both; `RA-MDST-TP` = both + type-preserving synthesis.

In [ ]:
def section5():
    perf = MAIN[0]
    order = ["MDST", "Abl-ExclusiveOnly", "Abl-InterpOnly", "RA-MDST", "RA-MDST-TP"]
    d = perf[perf.sampler.isin(order) & perf.classifier.isin(ABLATION_CLASSIFIERS)]
    t = (d.groupby(["classifier", "sampler"])[["accuracy", "f1_macro", "balanced_accuracy"]].agg(["mean", "std"]) * 100).round(2)
    t.columns = ["_".join(c) for c in t.columns]
    t = t.reset_index(); t["sampler"] = pd.Categorical(t.sampler, order, ordered=True); t = t.sort_values(["classifier", "sampler"])
    save_table(t, "T05_ablation_protocolB.csv")
    print(t.to_string(index=False))
    SUMMARY["ablation_protocolB"] = t.astype({"sampler": str}).to_dict("records")
    fig, axes = plt.subplots(1, len(ABLATION_CLASSIFIERS), figsize=(4.2 * len(ABLATION_CLASSIFIERS), 3.6), sharey=True)
    for ax, c_name in zip(axes, ABLATION_CLASSIFIERS):
        tt = t[t.classifier == c_name]
        ax.bar(range(len(tt)), tt.f1_macro_mean, yerr=tt.f1_macro_std / math.sqrt(MAIN_REPEATS * N_SPLITS),
               color=[SAMPLER_COLORS[s] for s in tt.sampler.astype(str)], width=0.65, capsize=3)
        ax.set_xticks(range(len(tt))); ax.set_xticklabels(["MDST", "Excl.\nonly", "Interp.\nonly", "RA-MDST", "RA-MDST\n-TP"], fontsize=9)
        ax.set_title(c_name, fontsize=11)
        lo = tt.f1_macro_mean.min(); ax.set_ylim(max(0, lo - 8), tt.f1_macro_mean.max() + 4)
    axes[0].set_ylabel("Macro-F1 (%) ± s.e.")
    savefig(fig, "F05_ablation_protocolB")
    return t

run_section("Section 5 — ablation", section5) if MAIN is not None else None

## Section 6 — Validity of synthetic values (type preservation)
Percentage of synthetic values that cannot occur in real data: a binary variable not equal to 0/1, a label-encoded category or ordinal
level never observed, or a continuous value outside the observed range; plus the share of synthetic rows that break a one-hot / mutually-exclusive group.

In [ ]:
def section6():
    val = MAIN[1]
    cols = ["pct_invalid_values", "pct_rows_with_invalid", "pct_invalid_binary", "pct_invalid_nominal",
            "pct_invalid_ordinal", "pct_invalid_continuous", "pct_rows_group_violation"]
    t = val.groupby("sampler")[cols].mean().round(2)
    t = t.loc[[s for s in SAMPLERS_MAIN + ABLATION_SAMPLERS if s in t.index]]
    save_table(t.reset_index(), "T06a_validity_summary.csv")
    print(t.to_string())
    SUMMARY["validity"] = t.reset_index().to_dict("records")
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.bar(t.index, t.pct_invalid_values, color=[SAMPLER_COLORS[s] for s in t.index], width=0.65)
    for i, v in enumerate(t.pct_invalid_values):
        ax.text(i, v + 1, f"{v:.1f}", ha="center", fontsize=9)
    ax.set_ylabel("Invalid synthetic values (%)"); ax.set_xticks(range(len(t))); ax.set_xticklabels(t.index, rotation=35, ha="right")
    ax.set_title("Share of synthetic feature values that cannot occur in the real data", loc="left", fontsize=11)
    savefig(fig, "F06_invalid_values")
    return t

run_section("Section 6 — validity", section6) if MAIN is not None else None

## Section 7 — Direct diversity, fidelity and separability metrics
Computed in each Protocol B training fold on the scaled RFE space, per minority class, then averaged:
**unique_ratio** (distinct synthetic rows), **exact_dup_of_real**, **near_dup_of_real / near_dup_within_syn** (distance < 0.1 × median real nearest-neighbour distance),
**rel_nn_dist_to_real** (novelty), **rel_pairwise_spread** (inter-sample distance relative to the real class), **coverage / density** (Naeem et al., 2020),
**intrusion_rate** (nearest real neighbour belongs to another class), **knn5_purity**, and the **silhouette** of the augmented training set (class separability).

In [ ]:
def section7():
    div = MAIN[2]
    per = div[div["class"] != "ALL"]
    cols = ["unique_ratio", "exact_dup_of_real", "near_dup_of_real", "near_dup_within_syn", "rel_nn_dist_to_real",
            "rel_pairwise_spread", "coverage", "density", "intrusion_rate", "knn5_purity"]
    w = per.assign(**{c: per[c] * per.n_syn for c in cols})
    t = (w.groupby("sampler")[cols].sum().div(w.groupby("sampler").n_syn.sum(), axis=0))
    sil = div[div["class"] == "ALL"].groupby("sampler").silhouette_augmented.mean()
    t["silhouette_augmented"] = sil
    t = t.loc[[s for s in SAMPLERS_MAIN + ABLATION_SAMPLERS if s in t.index]].round(4)
    save_table(t.reset_index(), "T07a_diversity_summary.csv")
    save_table(per.groupby(["sampler", "class"])[cols].mean().round(4).reset_index(), "T07b_diversity_by_class.csv")
    print(t.to_string())
    SUMMARY["diversity"] = t.reset_index().to_dict("records")
    show = ["unique_ratio", "near_dup_within_syn", "rel_pairwise_spread", "rel_nn_dist_to_real", "coverage", "intrusion_rate"]
    labels = ["Unique synthetic rows", "Near-duplicates among synthetic", "Pairwise spread (rel. to real)",
              "Distance to nearest real (rel.)", "Coverage of real class", "Intrusion into other classes"]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
    for ax, c, lab in zip(axes.ravel(), show, labels):
        ax.bar(range(len(t)), t[c], color=[SAMPLER_COLORS[s] for s in t.index], width=0.65)
        ax.set_xticks(range(len(t))); ax.set_xticklabels(t.index, rotation=45, ha="right", fontsize=8)
        ax.set_title(lab, fontsize=10, loc="left")
    fig.tight_layout()
    savefig(fig, "F07_diversity_metrics")
    return t

run_section("Section 7 — diversity", section7) if MAIN is not None else None

## Section 8 — Sensitivity analysis (one factor at a time + k × λ grid), Protocol B, Linear SVM
Defaults: k = 3, λ ∈ [0.3, 0.7], RFE = 100 features, C = 1. Each factor is varied while the others stay at default.
`SENS_REPEATS` × 5 folds.

In [ ]:
def sens_configs():
    base = dict(k=K_NEIGH, lam=LAM, n=N_RFE, C=C_SVM)
    cfgs = []
    def add(sampler, **kw):
        c = dict(base)
        if sampler == "SMOTE":
            c["k"] = IMB_K                      # SMOTE keeps its library default unless k itself is varied
        c.update(kw); c["sampler"] = sampler; cfgs.append(c)
    for k in SENS_K:
        for s in ["SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]: add(s, k=k, factor="k")
    for lam in SENS_LAM:
        for s in ["RA-MDST", "RA-MDST-TP"]: add(s, lam=lam, factor="lambda")
    for n in SENS_N:
        for s in ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]: add(s, n=n, factor="n_features")
    for C in SENS_C:
        for s in ["None", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]: add(s, C=C, factor="C")
    for k in SENS_K:
        for lam in SENS_LAM:
            for s in ["RA-MDST", "RA-MDST-TP"]: add(s, k=k, lam=lam, factor="grid_k_lambda")
    return cfgs

def sens_task(r, f, tr, te, seed, cfgs):
    ctx = FoldCtx(X_RAW, Y, tr, te, seed)
    out, cache = [], {}
    for c in cfgs:
        Xtr, Xte, ti = ctx.data(c["n"])
        key = (c["sampler"], c["n"], c["k"], c["lam"])
        if key not in cache:
            cache[key] = oversample(c["sampler"], Xtr, ctx.y_tr, ti, seed, k=c["k"], lam=c["lam"])
        Xa, ya, _ = cache[key]
        pred = get_model("LinearSVM", seed, C=c["C"], max_iter=FAST_SVC_MAX_ITER).fit(Xa, ya).predict(Xte)
        m = metrics(ctx.y_te, pred)
        out.append({**{k_: (str(v) if k_ == "lam" else v) for k_, v in c.items()}, "repeat": r, "fold": f, **m})
    return out

def section8():
    cfgs = sens_configs()
    res = Parallel(n_jobs=N_JOBS, verbose=5, timeout=TASK_TIMEOUT_SEC)(delayed(sens_task)(*t, cfgs) for t in outer_splits(Y, SENS_REPEATS))
    S = pd.DataFrame([x for r_ in res for x in r_])
    save_table(S, "T08_sensitivity_fold_level.csv")
    keycols = ["factor", "sampler", "k", "lam", "n", "C"]
    agg = S.groupby(keycols)[["f1_macro", "balanced_accuracy", "accuracy"]].agg(["mean", "std"]) * 100
    agg.columns = ["_".join(c) for c in agg.columns]
    agg = agg.reset_index().round(2)
    save_table(agg, "T08a_sensitivity_summary.csv")
    fac = [("k", "k", "Neighbourhood size k"), ("lambda", "lam", "λ range"), ("n_features", "n", "RFE features"), ("C", "C", "SVM C (log scale)")]
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.9))
    for ax, (fname, col, lab) in zip(axes, fac):
        d = agg[agg.factor == fname]
        for s_name, g in d.groupby("sampler"):
            g = g.copy()
            if col == "lam":
                g["x"] = g[col].map(lambda s: float(eval(s)[1]) - float(eval(s)[0])); g = g.sort_values("x")
                xs = g["x"]
            else:
                g = g.sort_values(col); xs = g[col]
            ax.errorbar(xs, g.f1_macro_mean, yerr=g.f1_macro_std / math.sqrt(SENS_REPEATS * N_SPLITS), marker="o", ms=4,
                        lw=1.8, capsize=2, color=SAMPLER_COLORS[s_name], label=s_name)
        if col == "C": ax.set_xscale("log")
        ax.set_xlabel(lab if col != "lam" else "Width of λ interval (0 = λ fixed at 0.5)")
        ax.set_title(lab, loc="left", fontsize=11)
    axes[0].set_ylabel("Macro-F1 (%) ± s.e.")
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    savefig(fig, "F08_sensitivity_ofat")
    g = agg[(agg.factor == "grid_k_lambda") & (agg.sampler == "RA-MDST-TP")].pivot(index="k", columns="lam", values="f1_macro_mean")
    if len(g):
        fig, ax = plt.subplots(figsize=(7, 3.8))
        im = ax.imshow(g.values, cmap="Blues", aspect="auto"); ax.grid(False)
        ax.set_xticks(range(g.shape[1])); ax.set_xticklabels(g.columns, rotation=30, ha="right")
        ax.set_yticks(range(g.shape[0])); ax.set_yticklabels(g.index); ax.set_ylabel("k"); ax.set_xlabel("λ range")
        for i in range(g.shape[0]):
            for j in range(g.shape[1]):
                ax.text(j, i, f"{g.values[i,j]:.1f}", ha="center", va="center", fontsize=9)
        fig.colorbar(im, ax=ax, label="Macro-F1 (%)")
        ax.set_title("RA-MDST-TP: k × λ grid (Linear SVM, Protocol B)", loc="left", fontsize=11)
        savefig(fig, "F08b_sensitivity_grid_k_lambda")
    rng_tab = agg[agg.factor != "grid_k_lambda"].groupby(["factor", "sampler"]).f1_macro_mean.agg(["min", "max"])
    rng_tab["range_pp"] = rng_tab["max"] - rng_tab["min"]
    save_table(rng_tab.reset_index().round(2), "T08b_sensitivity_ranges.csv")
    print(rng_tab.round(2).to_string())
    SUMMARY["sensitivity_ranges"] = rng_tab.reset_index().round(2).to_dict("records")
    return S

run_section("Section 8 — sensitivity", section8)

## Section 9 — Nested cross-validation
Outer: `NESTED_OUTER_REPEATS` × 5-fold. Inner: `NESTED_INNER`-fold stratified CV on the outer training rows only
(scaler, RFE and oversampling re-fitted inside every inner split). The inner loop selects the configuration with the best mean macro-F1;
it is refitted on the outer training rows and scored once on the outer test fold. Linear SVM.

In [ ]:
def sampler_grid(s_name):
    G = NESTED_GRID; out = []
    for n in G["n"]:
        for C in G["C"]:
            if s_name == "None":
                out.append(dict(n=n, C=C, k=None, lam=LAM)); continue
            for k in G["k"]:
                if s_name in ("RA-MDST", "RA-MDST-TP"):
                    for lam in G["lam"]:
                        out.append(dict(n=n, C=C, k=k, lam=lam))
                else:
                    out.append(dict(n=n, C=C, k=k, lam=LAM))
    return out

def eval_grid(ctx, s_name, grid, seed):
    scores, cache = [], {}
    for g in grid:
        Xtr, Xte, ti = ctx.data(g["n"])
        key = (g["n"], g["k"], g["lam"])
        if key not in cache:
            cache[key] = oversample(s_name, Xtr, ctx.y_tr, ti, seed, k=g["k"], lam=g["lam"])
        Xa, ya, _ = cache[key]
        pred = get_model("LinearSVM", seed, C=g["C"], max_iter=FAST_SVC_MAX_ITER).fit(Xa, ya).predict(Xte)
        scores.append(metrics(ctx.y_te, pred))
    return scores

def nested_task(r, f, tr, te, seed):
    outer = FoldCtx(X_RAW, Y, tr, te, seed)
    inner_cv = StratifiedKFold(n_splits=NESTED_INNER, shuffle=True, random_state=seed)
    inner_ctx = []
    for i, (itr, ite) in enumerate(inner_cv.split(np.zeros(len(tr)), Y[tr])):
        inner_ctx.append(FoldCtx(X_RAW, Y, tr[itr], tr[ite], seed * 10 + i))
    out = []
    for s_name in NESTED_SAMPLERS:
        grid = sampler_grid(s_name)
        inner_scores = np.zeros(len(grid))
        for i, ic in enumerate(inner_ctx):
            sc = eval_grid(ic, s_name, grid, seed * 10 + i)
            inner_scores += np.array([m["f1_macro"] for m in sc]) / len(inner_ctx)
        best = int(np.argmax(inner_scores)); g = grid[best]
        m = eval_grid(outer, s_name, [g], seed)[0]
        out.append({"repeat": r, "fold": f, "sampler": s_name, "best_n": g["n"], "best_C": g["C"], "best_k": g["k"],
                    "best_lam": str(g["lam"]) if s_name in ("RA-MDST", "RA-MDST-TP") else "", "inner_f1": inner_scores[best], **m})
    return out

def section9():
    res = Parallel(n_jobs=N_JOBS, verbose=10, timeout=TASK_TIMEOUT_SEC)(delayed(nested_task)(*t) for t in outer_splits(Y, NESTED_OUTER_REPEATS))
    Nst = pd.DataFrame([x for r_ in res for x in r_])
    save_table(Nst, "T09_nested_cv_fold_level.csv")
    t = (Nst.groupby("sampler")[["accuracy", "f1_macro", "balanced_accuracy", "inner_f1"]].agg(["mean", "std"]) * 100).round(2)
    t.columns = ["_".join(c) for c in t.columns]
    t = t.loc[[s for s in NESTED_SAMPLERS if s in t.index]]
    save_table(t.reset_index(), "T09a_nested_cv_summary.csv")
    chosen = (Nst.groupby("sampler")[["best_n", "best_C", "best_k", "best_lam"]]
              .agg(lambda s: s.astype(str).value_counts().to_dict()))
    save_table(chosen.reset_index(), "T09b_nested_cv_selected_hyperparameters.csv")
    print(t.to_string()); print(chosen.to_string())
    SUMMARY["nested"] = t.reset_index().to_dict("records")
    SUMMARY["nested_selected"] = chosen.reset_index().astype(str).to_dict("records")
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.bar(t.index, t.f1_macro_mean, yerr=t.f1_macro_std / math.sqrt(NESTED_OUTER_REPEATS * N_SPLITS),
           color=[SAMPLER_COLORS[s] for s in t.index], width=0.6, capsize=3)
    ax.set_ylabel("Outer-fold macro-F1 (%) ± s.e."); ax.set_ylim(max(0, t.f1_macro_mean.min() - 10), t.f1_macro_mean.max() + 5)
    ax.set_title("Nested CV (hyper-parameters tuned in the inner loop)", loc="left", fontsize=11)
    savefig(fig, "F09_nested_cv")
    return Nst

run_section("Section 9 — nested CV", section9)

## Section 10 — Partial balancing and real-to-synthetic ratios
Each minority class is oversampled to `n_c + α (n_max − n_c)`; α = 0 is no oversampling, α = 1 is full balancing (the paper's setting).

In [ ]:
def pb_task(r, f, tr, te, seed):
    ctx = FoldCtx(X_RAW, Y, tr, te, seed)
    Xtr, Xte, ti = ctx.data(N_RFE)
    out = []
    for alpha in [0.0] + PB_ALPHAS:
        tg = class_targets(ctx.y_tr, alpha)
        for s_name in (["None"] if alpha == 0 else PB_SAMPLERS):
            Xa, ya, _ = oversample(s_name, Xtr, ctx.y_tr, ti, seed, targets=tg)
            n_syn = len(ya) - len(ctx.y_tr)
            for c_name in PB_CLASSIFIERS:
                pred = get_model(c_name, seed).fit(Xa, ya).predict(Xte)
                out.append({"repeat": r, "fold": f, "alpha": alpha, "sampler": s_name, "classifier": c_name,
                            "n_real": len(ctx.y_tr), "n_syn": n_syn, "syn_share_pct": 100 * n_syn / len(ya),
                            "real_to_syn": (len(ctx.y_tr) / n_syn) if n_syn else np.inf,
                            **metrics(ctx.y_te, pred)})
    return out

def section10():
    res = Parallel(n_jobs=N_JOBS, verbose=5, timeout=TASK_TIMEOUT_SEC)(delayed(pb_task)(*t) for t in outer_splits(Y, PB_REPEATS))
    P = pd.DataFrame([x for r_ in res for x in r_])
    save_table(P, "T10_partial_balancing_fold_level.csv")
    t = P.groupby(["classifier", "sampler", "alpha"]).agg(
        n_syn=("n_syn", "mean"), syn_share_pct=("syn_share_pct", "mean"), real_to_syn=("real_to_syn", "mean"),
        f1_macro=("f1_macro", "mean"), f1_sd=("f1_macro", "std"), bal_acc=("balanced_accuracy", "mean"),
        accuracy=("accuracy", "mean")).reset_index()
    for c in ["f1_macro", "f1_sd", "bal_acc", "accuracy"]: t[c] *= 100
    save_table(t.round(3), "T10a_partial_balancing_summary.csv")
    print(t.round(2).to_string(index=False))
    SUMMARY["partial_balancing"] = t.round(2).to_dict("records")
    fig, axes = plt.subplots(1, len(PB_CLASSIFIERS), figsize=(4.4 * len(PB_CLASSIFIERS), 3.7), sharey=True)
    base = t[t.sampler == "None"]
    for ax, c_name in zip(axes, PB_CLASSIFIERS):
        b0 = base[base.classifier == c_name].f1_macro.values[0]
        for s_name in PB_SAMPLERS:
            d = t[(t.classifier == c_name) & (t.sampler == s_name)].sort_values("alpha")
            ax.plot([0] + d.alpha.tolist(), [b0] + d.f1_macro.tolist(), marker="o", ms=4, lw=1.8,
                    color=SAMPLER_COLORS[s_name], label=s_name)
        ax.set_xlabel("Balancing level α (0 = none, 1 = full)"); ax.set_title(c_name, fontsize=11)
    axes[0].set_ylabel("Macro-F1 (%)"); axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    savefig(fig, "F10_partial_balancing")
    return P

run_section("Section 10 — partial balancing", section10)

## Section 11 — Ordinal analysis
Labels are re-coded to the severity order in `SEVERITY_ORDER` (the original codes 0/1/2 are alphabetical, not ordinal).
Ordinal models: **Frank & Hall (2001)** cumulative-binary logistic regression and, if available, **mord LogisticAT** (all-threshold ordinal logit).
Nominal models (Linear SVM, logistic regression) are evaluated on the same folds. Ordinal metrics: MAE in severity steps, quadratic-weighted κ,
and the rate of two-step errors (e.g. "Moderate" predicted as "None").

In [ ]:
class FrankHallOrdinal(BaseEstimator, ClassifierMixin):
    def __init__(self, C=1.0, seed=SEED):
        self.C = C; self.seed = seed
    def fit(self, X, y_ord):
        self.levels_ = np.sort(np.unique(y_ord)); self.models_ = []
        for t in self.levels_[:-1]:
            self.models_.append(LogisticRegression(C=self.C, max_iter=5000, random_state=self.seed).fit(X, (y_ord > t).astype(int)))
        return self
    def predict_proba(self, X):
        gt = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models_])
        gt = np.minimum.accumulate(gt, axis=1)                     # enforce P(y>t) non-increasing in t
        P = np.column_stack([1 - gt[:, 0]] + [gt[:, i - 1] - gt[:, i] for i in range(1, gt.shape[1])] + [gt[:, -1]])
        return np.clip(P, 0, 1)
    def predict(self, X):
        return self.levels_[np.argmax(self.predict_proba(X), 1)]

INV_ORD = {v: k for k, v in ORD_MAP.items()}

def ord_task(r, f, tr, te, seed):
    ctx = FoldCtx(X_RAW, Y, tr, te, seed)
    Xtr, Xte, ti = ctx.data(N_RFE)
    out = []
    for s_name in ORD_SAMPLERS:
        Xa, ya, _ = oversample(s_name, Xtr, ctx.y_tr, ti, seed)
        yo = ordinal_codes(ya)
        models = {"LinearSVM (nominal)": ("nom", get_model("LinearSVM", seed)),
                  "LogisticRegression (nominal)": ("nom", get_model("LogisticRegression", seed)),
                  "Ordinal logistic (Frank-Hall)": ("ord", FrankHallOrdinal(1.0, seed))}
        if mord is not None:
            models["Ordinal logistic (mord AT)"] = ("ord", mord.LogisticAT(alpha=1.0))
        for m_name, (kind, mdl) in models.items():
            try:
                if kind == "nom":
                    pred = mdl.fit(Xa, ya).predict(Xte)
                else:
                    pred = np.array([INV_ORD[int(v)] for v in mdl.fit(Xa, yo).predict(Xte)])
                m = metrics(ctx.y_te, pred)
                m["two_step_error_rate"] = float(np.mean(np.abs(ordinal_codes(ctx.y_te) - ordinal_codes(pred)) == 2))
                out.append({"repeat": r, "fold": f, "sampler": s_name, "model": m_name, **m})
            except Exception as e:
                out.append({"repeat": r, "fold": f, "sampler": s_name, "model": m_name, "error": str(e)[:150]})
    return out

def section11():
    res = Parallel(n_jobs=N_JOBS, verbose=5, timeout=TASK_TIMEOUT_SEC)(delayed(ord_task)(*t) for t in outer_splits(Y, ORD_REPEATS))
    O = pd.DataFrame([x for r_ in res for x in r_])
    save_table(O, "T11_ordinal_fold_level.csv")
    cols = ["f1_macro", "balanced_accuracy", "accuracy", "ord_mae", "ord_qwk", "two_step_error_rate"]
    t = O.groupby(["model", "sampler"])[cols].mean().round(4).reset_index()
    save_table(t, "T11a_ordinal_summary.csv")
    print(t.to_string(index=False))
    SUMMARY["ordinal"] = t.to_dict("records")
    return O

run_section("Section 11 — ordinal analysis", section11)

## Section 12 — SHAP explanations (out-of-fold, Protocol B, repeat 0)
Models are trained on RA-MDST-TP–augmented training folds and explained **on the untouched real test rows only**.
* Linear SVM: one-vs-rest `LinearSVC` (C = 1) — exact linear SHAP values φ = w·(x − E[x]) (interventional, independent features).
* Random Forest: `shap.TreeExplainer`.
Importance = mean |SHAP| per original feature; features not selected by RFE in a fold contribute 0 for that fold.
**Stability** = number of the 5 folds in which a feature is among that fold's top-20.

In [ ]:
def section12():
    rows, top_sets = [], defaultdict(lambda: defaultdict(int))
    for r, f, tr, te, seed in outer_splits(Y, 1):
        ctx = FoldCtx(X_RAW, Y, tr, te, seed)
        Xtr, Xte, ti = ctx.data(N_RFE); sel = ctx.sel(N_RFE)
        Xa, ya, _ = oversample("RA-MDST-TP", Xtr, ctx.y_tr, ti, seed)
        lin = LinearSVC(C=1.0, dual=False, max_iter=20000, random_state=seed).fit(Xa, ya)
        phi_lin = lin.coef_[:, None, :] * (Xte - Xa.mean(0))[None, :, :]       # (class, n, p)
        phis = {"LinearSVM (OvR)": phi_lin}
        if shap is not None:
            rf = RandomForestClassifier(n_estimators=TREE_EST, random_state=seed, n_jobs=-1).fit(Xa, ya)
            sv = shap.TreeExplainer(rf).shap_values(Xte, check_additivity=False)
            sv = np.stack(sv, 0) if isinstance(sv, list) else np.moveaxis(np.asarray(sv), -1, 0)
            phis["RandomForest"] = sv
        for mname, ph in phis.items():
            for ci, c in enumerate(CLASSES):
                imp = np.abs(ph[ci]).mean(0)
                for j, fi in enumerate(sel):
                    rows.append({"fold": f, "model": mname, "class": CLASS_NAMES[c], "feature": FEATURES[fi],
                                 "mean_abs_shap": imp[j], "n_test": len(te)})
            overall = np.abs(ph).mean((0, 1))
            for fi in sel[np.argsort(-overall)[:20]]:
                top_sets[mname][FEATURES[fi]] += 1
    S = pd.DataFrame(rows)
    save_table(S, "T12_shap_fold_level.csv")
    nfold = S.fold.nunique()
    agg = (S.groupby(["model", "class", "feature"]).mean_abs_shap.sum() / nfold).reset_index()
    overall = (agg.groupby(["model", "feature"]).mean_abs_shap.mean()).reset_index()
    overall["top20_stability_folds"] = [top_sets[m][f_] for m, f_ in zip(overall.model, overall.feature)]
    overall = overall.sort_values(["model", "mean_abs_shap"], ascending=[True, False])
    save_table(overall, "T12a_shap_global_importance.csv")
    save_table(agg.sort_values(["model", "class", "mean_abs_shap"], ascending=[True, True, False]), "T12b_shap_by_class.csv")
    top = overall.groupby("model").head(15)
    print(top.round(4).to_string(index=False))
    SUMMARY["shap_top10"] = overall.groupby("model").head(10).round(4).to_dict("records")
    models = overall.model.unique()
    fig, axes = plt.subplots(1, len(models), figsize=(6.5 * len(models), 5.2))
    axes = np.atleast_1d(axes)
    for ax, mname in zip(axes, models):
        d = overall[overall.model == mname].head(15)[::-1]
        ax.barh(d.feature, d.mean_abs_shap, color="#2a78d6", height=0.65)
        for yv, (v, s_) in enumerate(zip(d.mean_abs_shap, d.top20_stability_folds)):
            ax.text(v, yv, f"  {s_}/{nfold}", va="center", fontsize=8, color="#52514e")
        ax.set_xlabel("Mean |SHAP| on held-out real rows"); ax.set_title(mname, loc="left", fontsize=11)
        ax.tick_params(axis="y", labelsize=8)
    fig.suptitle("Top-15 features (label = folds in which the feature is in the top-20)", x=0.02, ha="left", fontsize=11)
    fig.tight_layout()
    savefig(fig, "F12_shap_global")
    return overall

run_section("Section 12 — SHAP", section12)

## Section 13 — External benchmark datasets (R2 #2)
Public imbalanced datasets from `imblearn.datasets.fetch_datasets` (Lemaître et al., 2017; requires Internet).
Same Protocol B pipeline without RFE (all features, fold-internal scaling), Linear SVM (`LinearSVC`, C = 1, for speed) and Random Forest.
These are binary problems; they test whether RA-MDST behaves differently from MDST and SMOTE outside the AV data — they do not replace a second crash dataset.

To add your own second dataset (e.g. an updated CA DMV extract or NHTSA SGO reports), append `{"name": ..., "path": ..., "target": ...}` to `EXTRA_CSV` below.

In [ ]:
EXTRA_CSV = []      # e.g. [{"name": "nhtsa_sgo", "path": "/kaggle/input/sgo/sgo_clean.csv", "target": "severity"}]

def ext_task(name, X, y, r, f, tr, te, seed, feat_names):
    ctx = FoldCtx(X, y, tr, te, seed, names=feat_names)
    Xtr, Xte, ti = ctx.data(X.shape[1])
    out = []
    for s_name in ["None", "ROS", "SMOTE", "Borderline-SMOTE", "ADASYN", "MDST", "RA-MDST", "RA-MDST-TP"]:
        Xa, ya, meta = oversample(s_name, Xtr, ctx.y_tr, ti, seed)
        for c_name, mdl in [("LinearSVM", LinearSVC(C=1.0, dual=False, max_iter=20000, random_state=seed)),
                            ("RandomForest", RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=1))]:
            pred = mdl.fit(Xa, ya).predict(Xte)
            out.append({"dataset": name, "repeat": r, "fold": f, "sampler": s_name, "classifier": c_name,
                        "f1_macro": f1_score(ctx.y_te, pred, average="macro", zero_division=0),
                        "balanced_accuracy": balanced_accuracy_score(ctx.y_te, pred),
                        "sampler_failed": meta.get("failed", False)})
    return out

def section13():
    datasets = []
    try:
        from imblearn.datasets import fetch_datasets
        fd = fetch_datasets(filter_data=tuple(EXT_DATASETS), verbose=False)
        for nm, d in fd.items():
            yv = np.asarray(d.target); yv = (yv == yv.max()).astype(int)       # 1 = minority
            datasets.append((nm, np.asarray(d.data, float), yv, [f"f{i}" for i in range(d.data.shape[1])]))
    except Exception as e:
        print("[info] public benchmarks not available (Internet off?):", e)
    for spec in EXTRA_CSV:
        d = pd.read_csv(spec["path"]); yv = pd.factorize(d[spec["target"]], sort=True)[0]
        Xd = d.drop(columns=[spec["target"]]).apply(pd.to_numeric, errors="coerce").fillna(0)
        datasets.append((spec["name"], Xd.to_numpy(float), yv, list(Xd.columns)))
    if not datasets:
        print("No external datasets — skipped."); return None
    rows = []
    for nm, X, yv, fn in datasets:
        print(f"{nm}: n={len(yv)}, p={X.shape[1]}, class counts={np.bincount(yv).tolist()}")
        res = Parallel(n_jobs=N_JOBS, timeout=TASK_TIMEOUT_SEC)(delayed(ext_task)(nm, X, yv, *t, fn) for t in outer_splits(yv, EXT_REPEATS))
        rows += [x for r_ in res for x in r_]
    E = pd.DataFrame(rows)
    save_table(E, "T13_external_fold_level.csv")
    t = (E.groupby(["dataset", "classifier", "sampler"])[["f1_macro", "balanced_accuracy"]].mean() * 100).round(2).reset_index()
    save_table(t, "T13a_external_summary.csv")
    piv = t.pivot_table(index=["dataset", "classifier"], columns="sampler", values="f1_macro")
    ranks = piv.rank(axis=1, ascending=False).mean().sort_values()
    save_table(piv.reset_index(), "T13b_external_macroF1_matrix.csv")
    print(piv.round(2).to_string()); print("\nAverage rank (macro-F1):\n", ranks.round(2).to_string())
    SUMMARY["external_ranks"] = ranks.round(3).to_dict()
    SUMMARY["external_macroF1"] = piv.round(2).reset_index().astype(str).to_dict("records")
    return E

run_section("Section 13 — external benchmarks", section13)

## Section 14 — Summary file and ZIP
`RESULTS_SUMMARY.md` collects the key numbers; `PROTOCOL.md` documents the exact pipeline for the Methods section and the response letter.

In [ ]:
def df_md(records, max_rows=60):
    try:
        d = pd.DataFrame(records).head(max_rows)
        try:
            return d.to_markdown(index=False)
        except Exception:
            return "```\n" + d.to_string(index=False) + "\n```"
    except Exception:
        return str(records)

def section14():
    L = [f"# RA-MDST revision results ({RUN_MODE} run)\n", f"Generated: {time.strftime('%Y-%m-%d %H:%M')}\n",
         "## Environment\n", "```json\n" + json.dumps(VERSIONS, indent=1) + "\n```\n"]
    if "data" in SUMMARY:
        L += ["## Data\n", "```json\n" + json.dumps(SUMMARY["data"], indent=1, default=str) + "\n```\n"]
    if "overlap" in SUMMARY:
        o = SUMMARY["overlap"]
        L += ["## 1. Neighbourhood overlap\n",
              f"* All observations (Table 3 definition): **{o['all_obs']:.4f}**\n",
              f"* Minority observations only: {o['minority_obs']:.4f}\n",
              f"* Per generated sample inside Protocol B folds (old notebook's 0.7554): {o['per_generated_sample_protocolB']:.4f}\n\n",
              df_md(o["distribution"]), "\n"]
    for key, title in [("protocolA_seed42", "2. Protocol A reproduction (seed 42)"),
                       ("protocolA_seed_sweep", "2b. Protocol A seed sweep"),
                       ("protocolB_linearSVM", "3. Protocol B — Linear SVM (mean ± sd over all folds)"),
                       ("protocolB_macroF1_matrix", "3b. Protocol B macro-F1 (%) — classifier × sampler"),
                       ("stats_key", "4. Key pairwise statistics (macro-F1)"),
                       ("stats_survival", "4a. Comparisons surviving Holm correction (macro-F1)"),
                       ("friedman", "4b. Friedman / average ranks (macro-F1)"),
                       ("ablation_protocolB", "5. Ablation under Protocol B"),
                       ("validity", "6. Invalid synthetic values (%)"),
                       ("diversity", "7. Diversity / fidelity / separability"),
                       ("sensitivity_ranges", "8. Sensitivity: macro-F1 range per factor (pp)"),
                       ("nested", "9. Nested CV"), ("nested_selected", "9b. Selected hyper-parameters"),
                       ("partial_balancing", "10. Partial balancing"), ("ordinal", "11. Ordinal analysis"),
                       ("shap_top10", "12. SHAP top-10 features"), ("external_macroF1", "13. External benchmarks (macro-F1 %)")]:
        if key in SUMMARY:
            L += [f"## {title}\n", df_md(SUMMARY[key]), "\n"]
    if "external_ranks" in SUMMARY:
        L += ["Average ranks: " + json.dumps(SUMMARY["external_ranks"]) + "\n"]
    if ERRORS:
        L += ["## Sections that failed\n", "See ERRORS.txt\n"]
    open(os.path.join(OUT_DIR, "RESULTS_SUMMARY.md"), "w").write("\n".join(L))
    P = f"""# Experimental protocol (for the Methods section)

* Data: {N_OBS} records, {N_FEAT} encoded predictors, {len(CLASSES)} classes {SUMMARY.get('data', {}).get('class_counts')}.
* **Protocol B (primary, leakage-controlled)**: {MAIN_REPEATS} repeats × {N_SPLITS}-fold stratified CV (repeat r uses
  `StratifiedKFold(shuffle=True, random_state={SEED}+r)`). Within each outer fold, in this order:
  (1) StandardScaler fitted on the real training rows; (2) RFE (LinearSVC, C=1, step={RFE_STEP}) to {N_RFE} features fitted on the
  scaled real training rows; (3) oversampling of the training rows only, each minority class to the majority count;
  (4) classifier fitted on the augmented training rows; (5) evaluation on the untouched real test rows.
  Seeds: fold f of repeat 0 uses {SEED}+f (identical to the original submission); repeat r>0 uses {SEED}+100r+f for RFE, sampler and classifier.
* **Protocol A (reproduction only)**: scaler and RFE fitted on all {N_OBS} rows; oversampling of all rows to {3*max(np.bincount(Y))};
  10-fold stratified CV on the augmented set (test folds contain synthetic rows). Seeds {PROTO_A_SEEDS}.
* MDST (reconstruction): seed drawn uniformly with replacement from the class; synthetic sample = mean of the seed and its
  top-{K_NEIGH} Euclidean (even draws) or Manhattan (odd draws) neighbours. RA-MDST: Manhattan neighbours exclusive of the Euclidean ones,
  one candidate drawn from the 6-candidate pool, λ ~ U{LAM}, x = s + λ(q − s) (bounded pairwise interpolation).
  RA-MDST-TP: as RA-MDST, but binary and label-encoded nominal variables are copied from q with probability λ (else from s),
  mutually-exclusive binary groups are copied jointly, ordinal integers are interpolated and snapped to the nearest observed level.
* Baselines: imbalanced-learn {VERSIONS['imbalanced-learn']} RandomOverSampler, SMOTE, BorderlineSMOTE (borderline-1, m=10),
  ADASYN, SMOTE-NC (k = {IMB_K}), all inside the training fold.
* Statistics: Nadeau–Bengio corrected resampled t-test on fold-level differences; two-sided Wilcoxon signed-rank on repeat-level means;
  Holm and BH correction over the 10 classifiers; matched-pairs rank-biserial r and Cohen's d_z; cluster-bootstrap 95% CI (5000 resamples of repeats);
  Friedman test and Nemenyi critical difference on average ranks across classifiers.
* Nested CV: {NESTED_OUTER_REPEATS}×{N_SPLITS} outer folds, {NESTED_INNER}-fold inner CV, grid {NESTED_GRID}, selection by inner macro-F1.
* Software: {json.dumps({k: VERSIONS[k] for k in ['python','numpy','pandas','scipy','scikit-learn','imbalanced-learn','shap']})}
"""
    open(os.path.join(OUT_DIR, "PROTOCOL.md"), "w").write(P)
    # Manifest: one line per table with a content hash that ignores run-time columns (*_sec), so two runs can be compared
    import hashlib
    man = []
    for fn in sorted(os.listdir(TAB_DIR)):
        if not fn.endswith(".csv"): continue
        t = pd.read_csv(os.path.join(TAB_DIR, fn))
        t = t[[c for c in t.columns if not str(c).endswith("_sec")]]
        h = hashlib.sha256(t.round(6).to_csv(index=False).encode()).hexdigest()
        man.append({"file": "tables/" + fn, "rows": len(t), "cols": t.shape[1], "sha256_without_timing": h})
    for sub in ["figures", "paper_figures"]:
        d_ = os.path.join(OUT_DIR, sub)
        for fn in (sorted(os.listdir(d_)) if os.path.isdir(d_) else []):
            man.append({"file": f"{sub}/{fn}", "rows": np.nan, "cols": np.nan, "sha256_without_timing": ""})
    pd.DataFrame(man).to_csv(os.path.join(OUT_DIR, "MANIFEST.csv"), index=False)
    zpath = os.path.join(os.path.dirname(OUT_DIR), "RA_MDST_revision_results.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(OUT_DIR):
            for fn in files:
                full = os.path.join(root, fn); z.write(full, os.path.relpath(full, os.path.dirname(OUT_DIR)))
    print("Saved:", zpath)
    print(open(os.path.join(OUT_DIR, "RESULTS_SUMMARY.md")).read()[:6000])
    if ERRORS:
        print(f"\n{len(ERRORS)} section(s) failed — see ERRORS.txt")

## Section 16 — Type-preserving variants of SMOTE, Borderline-SMOTE and ADASYN (round-3 revision)
Each variant keeps its own algorithm (seeds, neighbours, interpolation steps) and changes only how values are generated:
continuous values are interpolated, ordinal values are interpolated and snapped to an observed level, binary and
label-encoded nominal values are copied from the seed or the neighbour (from the neighbour with probability equal to
the interpolation step), and mutually exclusive binary groups are copied jointly — the same rule as RA-MDST-TP.
This separates the effect of type preservation from that of the redundancy correction.

In [ ]:
from imblearn.over_sampling import SMOTE as _SMOTE, BorderlineSMOTE as _BSMOTE, ADASYN as _ADASYN
from sklearn.utils import check_random_state as _crs

def typed_rows(base, nb, steps, ti, rng):
    steps = np.asarray(steps, float).reshape(-1, 1)
    x = base + steps * (nb - base)
    for j in ti.ord_idx:
        x[:, j] = _snap(ti.valid_s[j], x[:, j])
    take = rng.random(x.shape) < steps
    for g in ti.groups:
        take[:, g["cols"]] = take[:, [g["cols"][0]]]
    d = ti.disc_idx
    x[:, d] = np.where(take[:, d], nb[:, d], base[:, d])
    return x

class _TPMixin:
    def _tp_rng(self):
        if not hasattr(self, "_rng_tp"):
            self._rng_tp = np.random.default_rng(int(self.random_state) + 7919)
        return self._rng_tp
    def _generate_samples(self, X, nn_data, nn_num, rows, cols, steps, y_type=None, y=None):
        return typed_rows(X[rows], nn_data[nn_num[rows, cols]], steps, self.ti_, self._tp_rng()).astype(X.dtype)

class SMOTE_TP(_TPMixin, _SMOTE):
    pass

class BorderlineSMOTE_TP(_TPMixin, _BSMOTE):
    pass

class ADASYN_TP(_TPMixin, _ADASYN):
    def _fit_resample(self, X, y):           # imbalanced-learn ADASYN, with the value-generation line made type-preserving
        self._validate_estimator()
        random_state = _crs(self.random_state)
        X_resampled, y_resampled = [X.copy()], [y.copy()]
        for class_sample, n_samples in self.sampling_strategy_.items():
            if n_samples == 0:
                continue
            idx = np.flatnonzero(y == class_sample); X_class = X[idx]
            self.nn_.fit(X)
            nns = self.nn_.kneighbors(X_class, return_distance=False)[:, 1:]
            n_neighbors = self.nn_.n_neighbors - 1
            ratio_nn = np.sum(y[nns] != class_sample, axis=1) / n_neighbors
            if not np.sum(ratio_nn):
                raise RuntimeError("ADASYN: no majority neighbours")
            ratio_nn /= np.sum(ratio_nn)
            n_gen = np.rint(ratio_nn * n_samples).astype(int); n_samples = np.sum(n_gen)
            if not n_samples:
                raise ValueError("ADASYN: nothing to generate")
            self.nn_.fit(X_class)
            nns = self.nn_.kneighbors(X_class, return_distance=False)[:, 1:]
            rows = np.repeat(np.arange(len(idx)), n_gen)
            cols = random_state.choice(n_neighbors, size=n_samples)
            steps = random_state.uniform(size=(n_samples, 1))
            X_new = typed_rows(X_class[rows], X_class[nns[rows, cols]], steps, self.ti_, self._tp_rng()).astype(X.dtype)
            X_resampled.append(X_new); y_resampled.append(np.full(n_samples, class_sample, dtype=y.dtype))
        return np.vstack(X_resampled), np.hstack(y_resampled)

TP_BASELINES = {"SMOTE-TP": SMOTE_TP, "Borderline-SMOTE-TP": BorderlineSMOTE_TP, "ADASYN-TP": ADASYN_TP}

def oversample_tp(name, X, y, ti, seed):
    targets = class_targets(y, 1.0)
    strat = {c: t for c, t in targets.items() if t > (y == c).sum()}
    min_n = min(int((y == c).sum()) for c in strat); kn = max(1, min(IMB_K, min_n - 1))
    kw = dict(sampling_strategy=strat, random_state=seed)
    if name == "ADASYN-TP":
        s = ADASYN_TP(n_neighbors=kn, **kw)
    elif name == "Borderline-SMOTE-TP":
        s = BorderlineSMOTE_TP(k_neighbors=kn, m_neighbors=10, **kw)
    else:
        s = SMOTE_TP(k_neighbors=kn, **kw)
    s.ti_ = ti
    Xa, ya = s.fit_resample(X, y)
    return np.asarray(Xa, float), np.asarray(ya, int)

def tp_fold_task(r, f, tr, te, seed, check=("SMOTE", "RA-MDST-TP")):
    ctx = FoldCtx(X_RAW, Y, tr, te, seed)
    Xtr, Xte, ti = ctx.data(N_RFE)
    ytr, yte = ctx.y_tr, ctx.y_te
    perf, val, div = [], [], []
    names = list(TP_BASELINES) + list(check)
    for s_name in names:
        if s_name in TP_BASELINES:
            Xa, ya = oversample_tp(s_name, Xtr, ytr, ti, seed)
        else:
            Xa, ya, _ = oversample(s_name, Xtr, ytr, ti, seed)
        base = {"repeat": r, "fold": f, "seed": seed, "sampler": s_name}
        syn = Xa[len(ytr):]
        if len(syn):
            v = validity(syn, ti); v.update(base); val.append(v)
            for d_ in diversity(Xtr, ytr, syn, ya[len(ytr):]):
                d_.update(base); div.append(d_)
        for c_name in CLASSIFIERS:
            model = get_model(c_name, seed)
            model.fit(Xa, ya)
            m = metrics(yte, model.predict(Xte)); m.update(base); m["classifier"] = c_name
            perf.append(m)
    return perf, val, div

def section16():
    tasks = outer_splits(Y, MAIN_REPEATS)
    res = Parallel(n_jobs=N_JOBS, verbose=5, timeout=TASK_TIMEOUT_SEC)(delayed(tp_fold_task)(*t) for t in tasks)
    perf = pd.DataFrame([x for r_ in res for x in r_[0]])
    val = pd.DataFrame([x for r_ in res for x in r_[1]])
    div = pd.DataFrame([x for r_ in res for x in r_[2]])
    save_table(perf, "T16_tp_baselines_fold_results.csv")
    save_table(val, "T16_tp_baselines_validity_fold_level.csv")
    save_table(div, "T16_tp_baselines_diversity_fold_level.csv")
    summarize_tp(perf, val, div, MAIN[0])
    return perf, val, div

def summarize_tp(perf, val, div, main_perf):
    """Combine the type-preserving baselines with the main Protocol B results (same folds) and test them."""
    tp_names = list(TP_BASELINES)
    allp = pd.concat([main_perf[main_perf.sampler.isin(SAMPLERS_MAIN)], perf[perf.sampler.isin(tp_names)]], ignore_index=True)
    order = ["None", "ROS", "SMOTE", "SMOTE-TP", "Borderline-SMOTE", "Borderline-SMOTE-TP", "ADASYN", "ADASYN-TP",
             "SMOTE-NC", "MDST", "RA-MDST", "RA-MDST-TP"]
    M = (allp.groupby(["classifier", "sampler"]).f1_macro.mean().unstack() * 100)[order].loc[CLASSIFIERS]
    ranks = M.rank(axis=1, ascending=False).mean()
    save_table(M.round(2).reset_index(), "T16a_macroF1_matrix_with_TP.csv")
    save_table(ranks.round(2).rename("avg_rank").reset_index(), "T16a_average_ranks_with_TP.csv")
    lin = allp[allp.classifier == "LinearSVM"].groupby("sampler")[["accuracy", "f1_macro", "balanced_accuracy"]].agg(["mean", "std"]) * 100
    save_table(lin.loc[order].round(2).reset_index(), "T16a_linearSVM_with_TP.csv")
    # validity and diversity (n_syn-weighted, as in Section 7)
    va = val.groupby("sampler")[["pct_invalid_values", "pct_rows_with_invalid", "pct_rows_group_violation"]].mean()
    per = div[div["class"] != "ALL"]
    cols = ["unique_ratio", "near_dup_within_syn", "rel_pairwise_spread", "rel_nn_dist_to_real", "intrusion_rate",
            "knn5_purity", "coverage", "density"]
    w = per.assign(**{c: per[c] * per.n_syn for c in cols})
    dv = w.groupby("sampler")[cols].sum().div(w.groupby("sampler").n_syn.sum(), axis=0)
    save_table(va.join(dv).round(4).reset_index(), "T16c_tp_validity_diversity.csv")
    # paired tests: RA-MDST-TP vs each TP baseline, and each TP baseline vs its original; Holm within each family of 30
    n_test = len(Y) / N_SPLITS; n_train = len(Y) - n_test
    fams = {"proposed_vs_TP": [("RA-MDST-TP", t) for t in tp_names],
            "TP_vs_original": [(t, t[:-3]) for t in tp_names]}
    rows = []
    for fam, comps in fams.items():
        for mtr in ["f1_macro", "balanced_accuracy", "accuracy"]:
            block = []
            for a, b in comps:
                for c_name in CLASSIFIERS:
                    A = allp[(allp.sampler == a) & (allp.classifier == c_name)].set_index(["repeat", "fold"])[mtr]
                    B = allp[(allp.sampler == b) & (allp.classifier == c_name)].set_index(["repeat", "fold"])[mtr]
                    d = (A - B).dropna()
                    mu, p, lo, hi = nb_corrected(d.values, n_train, n_test)
                    block.append({"family": fam, "metric": mtr, "comparison": f"{a} − {b}", "classifier": c_name,
                                  "delta_pp": 100 * mu, "nb_ci_low_pp": 100 * lo, "nb_ci_high_pp": 100 * hi,
                                  "p_nadeau_bengio": p, "cohens_dz_folds": mu / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan})
            b_ = pd.DataFrame(block); b_["p_holm_family30"] = holm(b_.p_nadeau_bengio.values); rows.append(b_)
    st = pd.concat(rows, ignore_index=True)
    save_table(st.round(5), "T16b_tp_statistics.csv")
    print(M.round(2).to_string()); print("\nAverage ranks:\n", ranks.sort_values().round(2).to_string())
    print("\nSignificant after Holm (family of 30):\n", st[st.p_holm_family30 < 0.05][["comparison", "metric", "classifier", "delta_pp", "p_holm_family30"]].to_string())
    SUMMARY["tp_baselines_ranks"] = ranks.round(2).to_dict()
    return M, ranks, st

In [ ]:
TP_RES = run_section("Section 16 — type-preserving baselines", section16) if MAIN is not None else None

## Section 17 — Sensitivity of the near-duplicate rate to its threshold (round-4 revision)
A synthetic sample is a near-duplicate when another synthetic sample of the same class lies closer than
f x (median nearest-neighbour distance among the real samples of that class). The article uses f = 0.1;
here f = 0.05, 0.1, 0.2 (and 0.3) are compared on the same synthetic samples (all 50 Protocol B training folds).

In [ ]:
ND_FRACS = [0.05, 0.1, 0.2, 0.3]
ND_SAMPLERS = ["ROS", "SMOTE", "Borderline-SMOTE", "ADASYN", "SMOTE-NC", "MDST", "RA-MDST", "RA-MDST-TP",
               "SMOTE-TP", "Borderline-SMOTE-TP", "ADASYN-TP"]

def nd_fold_task(r, f, tr, te, seed):
    ctx = FoldCtx(X_RAW, Y, tr, te, seed)
    Xtr, Xte, ti = ctx.data(N_RFE)
    ytr = ctx.y_tr
    rows = []
    for s_name in ND_SAMPLERS:
        if s_name in TP_BASELINES:
            Xa, ya = oversample_tp(s_name, Xtr, ytr, ti, seed)
        else:
            Xa, ya, _ = oversample(s_name, Xtr, ytr, ti, seed)
        S_all, yS = Xa[len(ytr):], ya[len(ytr):]
        for c in np.unique(yS):
            R = Xtr[ytr == c]; S = S_all[yS == c]
            if len(S) < 2 or len(R) < 6:
                continue
            dRR = cdist(R, R); np.fill_diagonal(dRR, np.inf); med = np.median(dRR.min(1))
            dSS = cdist(S, S); np.fill_diagonal(dSS, np.inf); nn = dSS.min(1)
            row = {"repeat": r, "fold": f, "sampler": s_name, "class": int(c), "n_syn": len(S)}
            for fr in ND_FRACS:
                row[f"near_dup_{fr}"] = float((nn < fr * med).mean())
            rows.append(row)
    return rows

def section17():
    tasks = outer_splits(Y, MAIN_REPEATS)
    res = Parallel(n_jobs=N_JOBS, verbose=0, timeout=TASK_TIMEOUT_SEC)(delayed(nd_fold_task)(*t) for t in tasks)
    D = pd.DataFrame([x for r_ in res for x in r_])
    save_table(D, "T17_near_duplicate_threshold_fold_level.csv")
    cols = [f"near_dup_{fr}" for fr in ND_FRACS]
    w = D.assign(**{c: D[c] * D.n_syn for c in cols})
    T = w.groupby("sampler")[cols].sum().div(w.groupby("sampler").n_syn.sum(), axis=0).loc[ND_SAMPLERS]
    ranks = T.rank(axis=0, method="min")
    out = T.round(4).join(ranks.add_suffix("_rank").astype(int))
    save_table(out.reset_index(), "T17a_near_duplicate_threshold_summary.csv")
    print(out.to_string())
    SUMMARY["near_dup_threshold"] = out.reset_index().to_dict("records")
    return out

In [ ]:
ND_RES = run_section("Section 17 — near-duplicate threshold", section17) if MAIN is not None else None

## Section 15 — article figures
Draws every result figure of the article from the tables above (identical tables give identical figures).
Output: `revision_results/paper_figures/`. The same code is in `paper_figures.py` for use outside the notebook.

In [ ]:
#!/usr/bin/env python3
_PF_DOC = """Draw the article's result figures from the CSV tables written by RA_MDST_revision.ipynb.

    python paper_figures.py --out revision_results        # writes revision_results/paper_figures/Fig*.pdf|png

Every figure is a pure function of the tables, so identical tables give identical figures.
The notebook calls make_all() at the end (Section 15)."""
import argparse, os, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SAMPLERS = ["None", "ROS", "SMOTE", "Borderline-SMOTE", "ADASYN", "SMOTE-NC", "MDST", "RA-MDST", "RA-MDST-TP"]
LABEL = {"None": "No oversampling", "ROS": "Random oversampling", "Borderline-SMOTE": "Borderline-SMOTE"}
COL = {"None": "#8c8c8c", "ROS": "#e87ba4", "SMOTE": "#eb6834", "Borderline-SMOTE": "#eda100", "ADASYN": "#e34948",
       "SMOTE-NC": "#008300", "MDST": "#1baf7a", "RA-MDST": "#2a78d6", "RA-MDST-TP": "#6250d6"}
CLF = ["LogisticRegression", "LinearSVM", "RBFSVM", "KNN", "DecisionTree", "RandomForest", "ExtraTrees",
       "GradientBoosting", "AdaBoost", "MLP"]
CLF_LABEL = {"LogisticRegression": "Logistic regression", "LinearSVM": "Linear SVM", "RBFSVM": "RBF SVM", "KNN": "k-NN",
             "DecisionTree": "Decision tree", "RandomForest": "Random forest", "ExtraTrees": "Extra trees",
             "GradientBoosting": "Gradient boosting", "AdaBoost": "AdaBoost", "MLP": "MLP"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True, "legend.frameon": False,
                     "savefig.dpi": 300, "pdf.fonttype": 42})

def _num(x):
    return float(str(x).split("±")[0])

def _save(fig, out, name):
    d = os.path.join(out, "paper_figures"); os.makedirs(d, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(d, f"{name}.{ext}"), bbox_inches="tight", metadata={"CreationDate": None} if ext == "pdf" else None)
    plt.close(fig)

def fig03_overlap(T, out):
    t = T("T01_overlap_distribution_all_observations.csv")
    n = t.n_observations.sum(); mean = (t.shared * t.n_observations).sum() / (3 * n)
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    b = ax.bar(t.shared, t.n_observations, color="#2a78d6", width=0.65)
    for x, v, p in zip(t.shared, t.n_observations, t.percent):
        ax.text(x, v + n * 0.01, f"{v}\n({p:.1f}%)", ha="center", va="bottom", fontsize=8.5)
    ax.set_xticks(t.shared, [f"{s}\n(score {['0', '1/3', '2/3', '1'][s]})" for s in t.shared])
    ax.set_xlabel("Shared neighbors between Euclidean and Manhattan top-3 sets"); ax.set_ylabel("Number of observations")
    ax.set_ylim(0, t.n_observations.max() * 1.28)
    ax.text(0.02, 0.97, f"Mean overlap = {mean:.4f} (n = {n})", transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round", fc="white", ec="#888"))
    _save(fig, out, "Fig03_overlap_distribution")

def fig06_protocols(T, out):
    a = T("T02a_protocolA_seed42_reproduction.csv").set_index("sampler")
    b = T("T03c_protocolB_LinearSVM_table.csv").set_index("sampler")
    s = [x for x in SAMPLERS if x in a.index and x in b.index]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, (ma, mr, mb, title) in zip(axes, [("accuracy", "acc_real_test_rows", "accuracy", "(a) Accuracy"),
                                             ("f1_macro", "f1_real_test_rows", "f1_macro", "(b) Macro-F1")]):
        x = np.arange(len(s)); w = 0.27
        ax.bar(x - w, a.loc[s, ma], w, color="#c9c9c9", label="Protocol A, all test rows")
        ax.bar(x, a.loc[s, mr], w, color="#8c8c8c", label="Protocol A, real test rows only")
        ax.bar(x + w, [_num(v) for v in b.loc[s, mb]], w, color="#2a78d6", label="Protocol B (leakage-controlled)")
        ax.set_xticks(x, [LABEL.get(v, v) for v in s], rotation=35, ha="right"); ax.set_title(title, loc="left")
        ax.set_ylim(0, 100)
    axes[0].set_ylabel("Linear SVM (%)")
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.04))
    _save(fig, out, "Fig06_protocolA_vs_B")

def fig07_heatmap(T, out):
    m = T("T03b_protocolB_macroF1_matrix.csv").set_index("classifier")
    s = [x for x in SAMPLERS if x in m.columns]; c = [x for x in CLF if x in m.index]
    M = m.loc[c, s]
    rk = T("T04b_friedman_average_ranks.csv"); rk = rk[rk.metric == "f1_macro"].set_index("sampler").avg_rank
    fig, ax = plt.subplots(figsize=(10, 4.6))
    im = ax.imshow(M.values, cmap="Blues", aspect="auto", vmin=np.nanmin(M.values) - 5, vmax=np.nanmax(M.values) + 2)
    best = M.values.argmax(axis=1)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.3, fontweight="bold" if j == best[i] else "normal",
                    color="white" if v > np.nanmean(M.values) + 5 else "#1f2328")
    SHORT = {"None": "None", "ROS": "ROS", "Borderline-SMOTE": "B-SMOTE"}
    ax.set_xticks(range(len(s)), [f"{SHORT.get(x, x)}\n(rank {rk.get(x, np.nan):.2f})" for x in s], fontsize=8)
    ax.set_xlabel("Oversampling method (average rank across classifiers; lower is better)")
    ax.set_yticks(range(len(c)), [CLF_LABEL[x] for x in c]); ax.grid(False)
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="Macro-F1 (%)")
    _save(fig, out, "Fig07_protocolB_macroF1_heatmap")

def fig08_forest(T, out):
    st = T("T04_pairwise_statistics.csv"); st = st[st.metric == "f1_macro"]
    comps = [x for x in ["RA-MDST − MDST", "RA-MDST-TP − MDST", "RA-MDST-TP − SMOTE", "RA-MDST − None", "RA-MDST-TP − None"]
             if x in set(st.comparison)]
    lo = st[st.comparison.isin(comps)].nb_ci_low_pp.min(); hi = st[st.comparison.isin(comps)].nb_ci_high_pp.max()
    rc = {"font.size": 12, "axes.titlesize": 12, "xtick.labelsize": 11, "ytick.labelsize": 12}
    with plt.rc_context(rc):
        fig, axes = plt.subplots(1, len(comps), figsize=(2.55 * len(comps), 4.6), sharey=True, sharex=True)
        _forest_panels(fig, axes, st, comps, lo, hi)
    _save(fig, out, "Fig08_forest_macroF1")

def _forest_panels(fig, axes, st, comps, lo, hi):
    for k, (ax, comp) in enumerate(zip(np.atleast_1d(axes), comps)):
        d = st[st.comparison == comp].set_index("classifier").reindex([x for x in CLF if x in set(st.classifier)])
        yy = np.arange(len(d))[::-1]; col = COL[comp.split(" − ")[0]]
        sig = d.p_nb_holm < 0.05
        ax.hlines(yy, d.nb_ci_low_pp, d.nb_ci_high_pp, color=col, lw=1.8)
        ax.scatter(d.delta_pp[sig], yy[sig.values], color=col, s=34, zorder=3)
        ax.scatter(d.delta_pp[~sig], yy[~sig.values], facecolor="white", edgecolor=col, s=34, zorder=3)
        ax.axvline(0, color="#52514e", lw=0.9)
        a_, b_ = comp.split(" − ")
        ax.set_title(f"({'abcde'[k]}) {a_}\n− {'no oversampling' if b_ == 'None' else b_}", loc="left")
        ax.set_xlim(lo - 2, hi + 2); ax.set_xlabel("Δ macro-F1 (pp)")
        ax.set_yticks(yy, [CLF_LABEL[x] for x in d.index])


def fig09_diversity(T, out):
    t = T("T07a_diversity_summary.csv").set_index("sampler")
    s = [x for x in ["ROS", "SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"] if x in t.index]
    panels = [("unique_ratio", "(a) Unique rows ↑"), ("near_dup_within_syn", "(b) Near-duplicates ↓"),
              ("rel_pairwise_spread", "(c) Spread ↑"), ("rel_nn_dist_to_real", "(d) Distance to real"),
              ("intrusion_rate", "(e) Intrusion ↓"), ("knn5_purity", "(f) 5-NN purity")]
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 5.0), sharey=True, gridspec_kw={"wspace": 0.3, "hspace": 0.45})
    axes = axes.ravel()
    for ax, (cname, title) in zip(axes, panels):
        v = t.loc[s, cname]
        ax.barh(range(len(s))[::-1], v, color=[COL[x] for x in s], height=0.65)
        for y, val in zip(range(len(s))[::-1], v):
            ax.text(val, y, f" {val:.3f}", va="center", fontsize=8.5)
        ax.set_title(title, fontsize=10, loc="left"); ax.set_xlim(0, max(1.0, v.max() * 1.35))
    for ax in (axes[0], axes[3]):
        ax.set_yticks(range(len(s))[::-1], [LABEL.get(x, x) for x in s])
    _save(fig, out, "Fig09_synthetic_diversity")

def fig10_sensitivity(T, out):
    t = T("T08a_sensitivity_summary.csv"); n_f = T("T08_sensitivity_fold_level.csv").groupby(["factor", "sampler"]).size().max()
    specs = [("k", "k", "(a) Neighborhood size k"), ("lambda", "lam", "(b) λ interval"),
             ("n_features", "n", "(c) RFE features"), ("C", "C", "(d) SVM C (log scale)")]
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharey=True)
    for ax, (fac, col, title) in zip(axes, specs):
        d = t[t.factor == fac]
        for s in [x for x in SAMPLERS if x in set(d.sampler)]:
            e = d[d.sampler == s].copy()
            if fac == "lambda":
                e["x"] = e[col].map(lambda z: float(re.findall(r"[\d.]+", str(z))[1]) - float(re.findall(r"[\d.]+", str(z))[0]))
            else:
                e["x"] = e[col].astype(float)
            e = e.sort_values("x"); reps = max(1, len(T("T08_sensitivity_fold_level.csv")[lambda q: (q.factor == fac) & (q.sampler == s)]) // max(len(e), 1))
            ax.errorbar(e.x, e.f1_macro_mean, yerr=e.f1_macro_std / np.sqrt(reps), color=COL[s], marker="o", ms=3.5, lw=1.4,
                        capsize=2, label=LABEL.get(s, s))
        ax.set_title(title, loc="left", fontsize=9)
        if fac == "C": ax.set_xscale("log")
        if fac == "lambda": ax.set_xlabel("interval width (centered at 0.5)")
    axes[0].set_ylabel("Macro-F1 (%)"); axes[-1].legend(fontsize=7.5, loc="lower right")
    _save(fig, out, "Fig10_sensitivity")

def fig11_partial(T, out):
    t = T("T10a_partial_balancing_summary.csv")
    clfs = [x for x in ["LinearSVM", "LogisticRegression", "RandomForest"] if x in set(t.classifier)]
    fig, axes = plt.subplots(1, len(clfs), figsize=(4.2 * len(clfs), 3.4))
    for k, (ax, c) in enumerate(zip(np.atleast_1d(axes), clfs)):
        d = t[t.classifier == c]; base = d[d.sampler == "None"].f1_macro
        for s in [x for x in SAMPLERS if x in set(d.sampler) and x != "None"]:
            e = pd.concat([d[d.sampler == "None"].assign(sampler=s), d[d.sampler == s]]).sort_values("alpha")
            ax.plot(e.alpha, e.f1_macro, marker="o", ms=3.5, color=COL[s], label=LABEL.get(s, s))
        if len(base): ax.axhline(base.iloc[0], color=COL["None"], ls="--", lw=1, label="No oversampling")
        ax.set_title(f"({'abc'[k]}) {CLF_LABEL[c]}", loc="left"); ax.set_xlabel("Balancing level α")
    np.atleast_1d(axes)[0].set_ylabel("Macro-F1 (%)"); np.atleast_1d(axes)[-1].legend(fontsize=7.5)
    _save(fig, out, "Fig11_partial_balancing")

def fig12_shap(T, out):
    t = T("T12a_shap_global_importance.csv")
    try:
        ident = {f for f in T("T00_feature_types.csv").feature if re.search(r"DATE|TIME|ADDRESS|DETAILS", str(f), re.I)}
    except Exception:
        ident = set()
    models = list(dict.fromkeys(t.model))[:2]
    fig, axes = plt.subplots(1, len(models), figsize=(6.4 * len(models), 4.6), gridspec_kw={"wspace": 0.55})
    for k, (ax, m) in enumerate(zip(np.atleast_1d(axes), models)):
        d = t[t.model == m].sort_values("mean_abs_shap", ascending=False).head(15)[::-1]
        ax.barh(d.feature, d.mean_abs_shap, color=["#b0b0b0" if f in ident else "#2a78d6" for f in d.feature])
        for y, (v, n) in enumerate(zip(d.mean_abs_shap, d.top20_stability_folds)):
            ax.text(v, y, f" {n}/5", va="center", fontsize=7.5)
        nm = "Linear SVM" if "Linear" in m else ("Random forest" if "Forest" in m else m)
        ax.set_title(f"({'ab'[k]}) {nm}", loc="left"); ax.set_xlabel("Mean |SHAP| on held-out real records")
        ax.set_xlim(0, d.mean_abs_shap.max() * 1.15)
    _save(fig, out, "Fig12_shap_importance")

def fig13_confusion(T, out):
    p = T("T03_protocolB_oof_predictions.csv")
    p = p[(p.repeat == 0) & (p.classifier == "LinearSVM")]
    names = {0: "Minor", 1: "Moderate", 2: "None"}
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8), gridspec_kw={"wspace": 0.45})
    for k, (ax, s) in enumerate(zip(axes, ["MDST", "RA-MDST"])):
        d = p[p.sampler == s]; labs = sorted(set(d.y_true))
        cm = pd.crosstab(d.y_true, d.y_pred).reindex(index=labs, columns=labs, fill_value=0).values
        r = cm / cm.sum(1, keepdims=True)
        ax.imshow(r, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
        for i in range(len(labs)):
            for j in range(len(labs)):
                ax.text(j, i, f"{100 * r[i, j]:.1f}%\n({cm[i, j]})", ha="center", va="center", fontsize=8.5,
                        color="white" if r[i, j] > 0.55 else "#1f2328")
        ax.set_xticks(range(len(labs)), [names.get(l, l) for l in labs]); ax.set_yticks(range(len(labs)), [names.get(l, l) for l in labs])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"({'ab'[k]}) {s}", loc="left")
    _save(fig, out, "Fig13_confusion_linearSVM")

def fig14_cost(T, out):
    f = T("T03_protocolB_fold_results.csv")
    f = f[(f.classifier == "LinearSVM") & f.sampler.isin(["MDST", "RA-MDST", "RA-MDST-TP", "SMOTE"])]
    g = f.groupby("sampler")[["sampling_sec", "fit_sec"]].mean().reindex(["SMOTE", "MDST", "RA-MDST", "RA-MDST-TP"]).dropna()
    fig, ax = plt.subplots(figsize=(5.4, 3.2)); x = np.arange(len(g)); w = 0.38
    ax.bar(x - w / 2, 1000 * g.sampling_sec, w, color="#2a78d6", label="Oversampling")
    ax.bar(x + w / 2, 1000 * g.fit_sec, w, color="#8c8c8c", label="Linear SVM training")
    ax.set_xticks(x, g.index); ax.set_ylabel("Mean time per fold (ms)"); ax.legend()
    _save(fig, out, "Fig14_computational_cost")

ALL = [fig03_overlap, fig06_protocols, fig07_heatmap, fig08_forest, fig09_diversity, fig10_sensitivity,
       fig11_partial, fig12_shap, fig13_confusion, fig14_cost]

def make_all(out):
    tabs = os.path.join(out, "tables")
    T = lambda name: pd.read_csv(os.path.join(tabs, name), keep_default_na=False, na_values=["", "nan", "NaN"])
    done, failed = [], []
    for fn in ALL:
        try:
            fn(T, out); done.append(fn.__name__)
        except FileNotFoundError as e:
            failed.append(f"{fn.__name__}: missing {os.path.basename(e.filename)}")
        except Exception as e:
            failed.append(f"{fn.__name__}: {type(e).__name__}: {e}")
    print("paper figures written:", ", ".join(done))
    for f in failed:
        print("  not drawn ->", f)
    return done, failed


_rc = dict(plt.rcParams)
make_all(OUT_DIR)
plt.rcParams.update(_rc)

In [ ]:
run_section("Section 14 — summary + zip", section14)